<a href="https://colab.research.google.com/github/smerarawal/IIT-BHU-Proj/blob/main/ps2_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import os
import pandas as pd
import numpy as np

def standardize_sequence(data, max_frames=200):
    """Pads or truncates data to a fixed length."""
    if data.shape[0] > max_frames:
        return data[:max_frames, :]
    else:
        padding = np.zeros((max_frames - data.shape[0], data.shape[1]))
        return np.vstack((data, padding))

def universal_loader(root_dir):
    X = []
    y = []

    for root, dirs, files in os.walk(root_dir):
        for file in files:
            file_path = os.path.join(root, file)

            label = 1 if 'correct' in root.lower() else 0

            try:
                if file.endswith('.csv'):
                    df = pd.read_csv(file_path)
                elif file.endswith('.json'):
                    df = pd.read_json(file_path)
                else:
                    continue

                # Extract numeric coordinate data (X, Y, Z)
                # Ensure you select only coordinates/angles, not timestamps or headers
                data = df.select_dtypes(include=[np.number]).values

                # Standardize to 200 frames
                seq = standardize_sequence(data)

                X.append(seq)
                y.append(label)

            except Exception as e:
                print(f"Skipping {file}: {e}")

    return np.array(X), np.array(y)

X_data, y_data = universal_loader('/content/drive/MyDrive/datasets')

print(f"Loading complete!")
print(f"Input Shape (Total Reps, Frames, Features): {X_data.shape}")

Loading complete!
Input Shape (Total Reps, Frames, Features): (1, 200, 0)


In [6]:
import os
import pandas as pd
import numpy as np

def universal_loader_verbose(root_dir):
    X, y = [], []
    count = 0

    if not os.path.exists(root_dir):
        print(f"CRITICAL ERROR: The path '{root_dir}' does not exist.")
        return np.array(X), np.array(y)

    for root, dirs, files in os.walk(root_dir):
        print(f"Checking folder: {root}")

        for file in files:
            if file.endswith(('.csv', '.json')):
                count += 1
                file_path = os.path.join(root, file)

                label = 1 if 'correct' in root.lower() else 0

    print(f"Total files found across all subfolders: {count}")
    return np.array(X), np.array(y)

X_data, y_data = universal_loader_verbose('/content/drive/MyDrive/datasets')

Checking folder: /content/drive/MyDrive/datasets
Checking folder: /content/drive/MyDrive/datasets/Incorrect Segmented Movements
Checking folder: /content/drive/MyDrive/datasets/Incorrect Segmented Movements/Vicon
Checking folder: /content/drive/MyDrive/datasets/Incorrect Segmented Movements/Vicon/Positions
Checking folder: /content/drive/MyDrive/datasets/Incorrect Segmented Movements/Vicon/Angles
Checking folder: /content/drive/MyDrive/datasets/Incorrect Segmented Movements/Kinect
Checking folder: /content/drive/MyDrive/datasets/Incorrect Segmented Movements/Kinect/Positions
Checking folder: /content/drive/MyDrive/datasets/Incorrect Segmented Movements/Kinect/Angles
Checking folder: /content/drive/MyDrive/datasets/Segmented Movements
Checking folder: /content/drive/MyDrive/datasets/Segmented Movements/Vicon
Checking folder: /content/drive/MyDrive/datasets/Segmented Movements/Vicon/Angles
Checking folder: /content/drive/MyDrive/datasets/Segmented Movements/Vicon/Positions
Checking folde

In [ ]:
import os
import pandas as pd
import numpy as np
import json

def load_and_standardize(data, max_frames=200):
    if data.shape[0] > max_frames:
        return data[:max_frames, :]
    else:
        padding = np.zeros((max_frames - data.shape[0], data.shape[1]))
        return np.vstack((data, padding))

def smart_loader(root_dir):
    X, y = [], []
    file_processed_count = 0
    total_files_found = 0

    if not os.path.exists(root_dir):
        print(f"Error: Root directory '{root_dir}' does not exist. Please check your path.")
        return np.array(X), np.array(y)

    print(f"Starting smart_loader for directory: {root_dir}")
    for root, dirs, files in os.walk(root_dir):
        print(f"Scanning directory: {root}")
        for file in files:
            total_files_found += 1
            file_path = os.path.join(root, file)
            print(f"  Found file: {file_path}")

            # UI-PRMD
            if 'Positions' in root and (file.endswith('.csv') or file.endswith('.txt')):
                print(f"  Attempting to load as UI-PRMD data (Positions): {file_path}")

                ang_filename = None
                if '_positions_inc' in file:
                    ang_filename = file.replace('_positions_inc', '_angles_inc')
                elif '_positions' in file:
                    ang_filename = file.replace('_positions', '_angles')

                if ang_filename is None:
                    print(f"    Skipping positions file with unknown naming convention: {file_path}")
                    continue

                angles_dir = root.replace('Positions', 'Angles')
                ang_path = os.path.join(angles_dir, ang_filename)

                if os.path.exists(ang_path):
                    print(f"    Found matching Angles file: {ang_path}")
                    try:
                        pos_df = pd.read_csv(file_path)
                        ang_df = pd.read_csv(ang_path)

                        # Combine pos and angles
                        combined = pd.concat([pos_df, ang_df], axis=1)
                        data = combined.select_dtypes(include=[np.number]).values

                        if data.size > 0:
                            X.append(load_and_standardize(data))
                            y.append(1 if 'Correct' in root else 0)
                            file_processed_count += 1
                            print(f"    Successfully processed UI-PRMD pair: {file}")
                        else:
                            print(f"    Warning: No numeric data found after combining for UI-PRMD pair: {file_path}")
                    except Exception as e:
                        print(f"    Error processing UI-PRMD pair {file_path} and {ang_path}: {e}")
                else:
                    print(f"    No matching Angles file found at expected path: {ang_path}. Skipping this Positions file.")
                    if not os.path.exists(angles_dir):
                        print(f"    Note: The expected Angles directory '{angles_dir}' does not exist.")
                    else:
                        print(f"    Note: Angles directory '{angles_dir}' exists, but file '{ang_filename}' was not found.")

            # UCO Phyrehab
            elif 'uco-phyrehab' in root.lower() and file.endswith('.json'):
                print(f"  Attempting to load as UCO Phyrehab JSON: {file_path}")
                try:
                    with open(file_path, 'r') as f:
                        data_raw = json.load(f)
                        df = pd.json_normalize(data_raw)
                        data = df.select_dtypes(include=[np.number]).values

                        if data.size > 0:
                            X.append(load_and_standardize(data))
                            y.append(1 if 'correct' in root.lower() else 0)
                            file_processed_count += 1
                            print(f"    Successfully processed UCO Phyrehab JSON: {file}")
                        else:
                            print(f"    Warning: No numeric data found in UCO Phyrehab JSON: {file_path}")
                except Exception as e:
                    print(f"    Error processing UCO Phyrehab JSON {file_path}: {e}")

            elif file.endswith('.json'):
                print(f"  Attempting to load as general JSON: {file_path}")
                try:
                    with open(file_path, 'r') as f:
                        data_raw = json.load(f)
                        df = pd.json_normalize(data_raw)
                        data = df.select_dtypes(include=[np.number]).values

                        if data.size > 0:
                            X.append(load_and_standardize(data))
                            y.append(1 if 'correct' in root.lower() else 0) # Default label if not explicitly in path
                            file_processed_count += 1
                            print(f"    Successfully processed general JSON: {file}")
                        else:
                            print(f"    Warning: No numeric data found in general JSON: {file_path}")
                except Exception as e:
                    print(f"    Error processing general JSON {file_path}: {e}")

            else:
                print(f"  File not matched by any specific loader rule: {file_path}. Skipping.")

    print(f"\n--- smart_loader Summary ---")
    print(f"Total files encountered during walk: {total_files_found}")
    print(f"Total files successfully processed: {file_processed_count}")
    print(f"Final X list length: {len(X)}")
    print(f"Final y list length: {len(y)}")

    return np.array(X), np.array(y)

X, y = smart_loader('/content/drive/MyDrive/datasets')
print(f"Final Shape: {X.shape}")

Streaming output truncated to the last 5000 lines.
  Attempting to load as UI-PRMD data (Positions): /content/drive/MyDrive/datasets/Segmented Movements/Vicon/Positions/m04_s05_e04_positions.txt
    Found matching Angles file: /content/drive/MyDrive/datasets/Segmented Movements/Vicon/Angles/m04_s05_e04_angles.txt
    Successfully processed UI-PRMD pair: m04_s05_e04_positions.txt
  Found file: /content/drive/MyDrive/datasets/Segmented Movements/Vicon/Positions/m04_s09_e01_positions.txt
  Attempting to load as UI-PRMD data (Positions): /content/drive/MyDrive/datasets/Segmented Movements/Vicon/Positions/m04_s09_e01_positions.txt
    Found matching Angles file: /content/drive/MyDrive/datasets/Segmented Movements/Vicon/Angles/m04_s09_e01_angles.txt
    Successfully processed UI-PRMD pair: m04_s09_e01_positions.txt
  Found file: /content/drive/MyDrive/datasets/Segmented Movements/Vicon/Positions/m05_s07_e01_positions.txt
  Attempting to load as UI-PRMD data (Positions): /content/drive/MyDriv

In [1]:
import zipfile, json, os

# UI-PRMD
with zipfile.ZipFile('Segmented Movements.zip', 'r') as z:
    # show first 30 paths
    for p in sorted(z.namelist())[:30]:
        print(p)

FileNotFoundError: [Errno 2] No such file or directory: 'Segmented Movements.zip'

In [ ]:
# UCO
with open('dataset_3d_with_angles.json') as f:
    data = json.load(f)

# check top-level keys and one sample
print(type(data))
if isinstance(data, list):
    print(len(data), "samples")
    print(data[0].keys())
elif isinstance(data, dict):
    print(data.keys())

In [2]:
import os

base = '/content/drive/MyDrive/datasets'

for folder in [
    'Segmented Movements/Vicon/Angles',
    'Segmented Movements/Kinect/Angles',
    'Incorrect Segmented Movements/Vicon/Angles',
    'Incorrect Segmented Movements/Kinect/Angles',
]:
    path = os.path.join(base, folder)
    files = sorted(os.listdir(path))[:10]
    print(f"\n── {folder}")
    for f in files:
        print(f"   {f}")


── Segmented Movements/Vicon/Angles
   m01_s01_e01_angles.txt
   m01_s01_e02_angles.txt
   m01_s01_e03_angles.txt
   m01_s01_e04_angles.txt
   m01_s01_e05_angles.txt
   m01_s01_e06_angles.txt
   m01_s01_e07_angles.txt
   m01_s01_e08_angles.txt
   m01_s01_e09_angles.txt
   m01_s01_e10_angles.txt

── Segmented Movements/Kinect/Angles
   m01_s01_e01_angles.txt
   m01_s01_e02_angles.txt
   m01_s01_e03_angles.txt
   m01_s01_e04_angles.txt
   m01_s01_e05_angles.txt
   m01_s01_e06_angles.txt
   m01_s01_e07_angles.txt
   m01_s01_e08_angles.txt
   m01_s01_e09_angles.txt
   m01_s01_e10_angles.txt

── Incorrect Segmented Movements/Vicon/Angles
   m01_s01_e01_angles_inc.txt
   m01_s01_e02_angles_inc.txt
   m01_s01_e03_angles_inc.txt
   m01_s01_e04_angles_inc.txt
   m01_s01_e05_angles_inc.txt
   m01_s01_e06_angles_inc.txt
   m01_s01_e07_angles_inc.txt
   m01_s01_e08_angles_inc.txt
   m01_s01_e09_angles_inc.txt
   m01_s01_e10_angles_inc.txt

── Incorrect Segmented Movements/Kinect/Angles
   m01_s01

In [5]:
import numpy as np

sample_path = '/content/drive/MyDrive/datasets/Segmented Movements/Vicon/Angles/m01_s01_e01_angles.txt' # Corrected to an existing file

data = np.loadtxt(sample_path, delimiter=',') # Added delimiter argument
print("shape:", data.shape)
print("first 3 rows:\n", data[:3])

shape: (191, 117)
first 3 rows:
 [[ 2.40387e+00  1.34607e+01  6.81939e-01 -1.34821e+01  2.30195e+00
  -9.57318e-01 -1.34821e+01 -2.30195e+00  9.57318e-01  1.09568e+01
  -2.01320e+00 -2.58148e-01  1.09568e+01  2.01320e+00  2.58148e-01
   8.25139e+01  5.16290e+00  3.94019e+01 -7.91178e+01  3.79335e+00
  -3.44421e+01 -1.20361e+01 -2.10244e+00 -3.36102e+00 -1.20361e+01
   2.10244e+00  3.36102e+00  1.80399e+02  1.18147e+00 -3.93044e+00
   2.49911e+00  4.28800e-01 -7.59822e-01  2.49911e+00 -4.28800e-01
   7.59822e-01 -1.29987e+00  1.45634e+01  4.40423e+00  1.45037e+01
  -1.84128e+00 -4.19397e+00  1.45037e+01  1.84128e+00  4.19397e+00
   2.72417e+01 -2.06503e+01 -2.27716e+01  2.25690e+01 -1.99894e+01
  -1.68547e+01  1.78048e+01 -1.56449e+01  2.92643e+01 -1.89041e+01
  -1.16259e+01 -1.35194e+01  2.03797e+01 -9.86000e-02 -8.32244e+00
   2.19423e+01  5.64107e+00 -1.63299e+01  1.27721e+01  3.26811e+00
   4.01401e+01 -1.17109e+01  7.66805e+00 -3.35513e+01  1.62887e+00
  -2.18909e+00  1.55109e+01  

In [6]:
import json

with open('/content/drive/MyDrive/datasets/dataset_3d_with_angles.json') as f:
    uco = json.load(f)

print(type(uco))
if isinstance(uco, list):
    print("num samples:", len(uco))
    print("keys:", uco[0].keys())
    print("one sample:\n", uco[0])
elif isinstance(uco, dict):
    print("top keys:", list(uco.keys())[:10])

<class 'dict'>
top keys: ['data']


In [9]:
import json
with open('/content/drive/MyDrive/datasets/dataset_3d_with_angles.json') as f:
    uco = json.load(f)

samples = uco['data']
print("num samples:", len(samples))
print("keys in one sample:", samples[0].keys())
print("\none sample:")
import pprint
pprint.pprint(samples[0])

num samples: 432
keys in one sample: dict_keys(['folder', 'exercise', 'position', 'side', 'body', 'n_frames', 'n_valid_frames', 'angle_mean', 'angle_min', 'angle_max', 'frames'])

one sample:
{'angle_max': 174.4944574153883,
 'angle_mean': 149.6620218058612,
 'angle_min': 104.53360804897125,
 'body': 'lower',
 'exercise': '01',
 'folder': '0',
 'frames': [{'id': 5,
             'joints': {'angle': 122.1981234681987,
                        'l_ankle': {'x': '0.703331',
                                    'y': '-0.148558',
                                    'z': '-1.78596'},
                        'l_hip': {'x': '1.2086',
                                  'y': '-0.65282',
                                  'z': '-1.28515'},
                        'l_knee': {'x': '0.785226',
                                   'y': '-0.554628',
                                   'z': '-1.61831'}}},
            {'id': 6,
             'joints': {'angle': 122.63581576732606,
                        'l_ankle

In [10]:
import os

base = '/content/drive/MyDrive/datasets/Segmented Movements/Vicon/Angles'
files = sorted(os.listdir(base))
print("total files:", len(files))
print("first 5:", files[:5])
print("last 5:", files[-5:])

movements = sorted(set(f.split('_')[0] for f in files))
subjects  = sorted(set(f.split('_')[1] for f in files))
print("movements:", movements)
print("subjects:", subjects)

total files: 990
first 5: ['m01_s01_e01_angles.txt', 'm01_s01_e02_angles.txt', 'm01_s01_e03_angles.txt', 'm01_s01_e04_angles.txt', 'm01_s01_e05_angles.txt']
last 5: ['m10_s10_e06_angles.txt', 'm10_s10_e07_angles.txt', 'm10_s10_e08_angles.txt', 'm10_s10_e09_angles.txt', 'm10_s10_e10_angles.txt']
movements: ['m01', 'm02', 'm03', 'm04', 'm05', 'm06', 'm07', 'm08', 'm09', 'm10']
subjects: ['s01', 's02', 's03', 's04', 's05', 's06', 's07', 's08', 's09', 's10']


In [11]:
import pandas as pd
import os

# check kinect positions
kp = pd.read_csv(
    '/content/drive/MyDrive/datasets/Segmented Movements/Kinect/Positions/m01_s01_e01_positions.txt',
    header=None
)
print("Kinect Positions shape:", kp.shape)
print(kp.head(2))

# check vicon positions
vp = pd.read_csv(
    '/content/drive/MyDrive/datasets/Segmented Movements/Vicon/Positions/m01_s01_e01_positions.txt',
    header=None
)
print("\nVicon Positions shape:", vp.shape)
print(vp.head(2))

Kinect Positions shape: (73, 66)
        0         1          2        3         4        5    6         7   \
0 -3.50280  85.59755 -235.88153 -0.00001  27.80241  0.00002  0.0  20.22640   
1 -3.48681  85.61602 -235.91014 -0.00001  27.79366  0.00002  0.0  20.22258   

        8    9   ...       56   57        58       59   60        61       62  \
0  0.00002  0.0  ...  3.57008  0.0 -36.16375 -0.00002  0.0 -38.82933 -0.00002   
1  0.00002  0.0  ...  3.57008  0.0 -36.17269 -0.00002  0.0 -38.89996 -0.00002   

    63       64        65  
0  0.0  0.00001  12.33617  
1  0.0  0.00001  12.21943  

[2 rows x 66 columns]

Vicon Positions shape: (191, 117)
       0        1        2        3        4        5        6        7    \
0 -138.008  89.4526  1529.73 -133.959 -29.7478  1525.73 -290.812  92.9676   
1 -135.122  89.5869  1526.20 -131.102 -29.6175  1522.23 -287.970  93.1370   

       8        9    ...      107      108      109      110      111  \
0  1568.41 -284.631  ...  323.394 -423.69

In [12]:
# also check incorrect kinect positions exist
inc_kp = pd.read_csv(
    '/content/drive/MyDrive/datasets/Incorrect Segmented Movements/Kinect/Positions/m01_s01_e01_positions_inc.txt',
    header=None
)
print("Incorrect Kinect Positions shape:", inc_kp.shape)
print(inc_kp.head(2))

Incorrect Kinect Positions shape: (84, 66)
        0         1          2    3         4        5        6         7   \
0 -5.56359  86.35738 -237.26701  0.0  27.96061 -0.00002 -0.00001  20.40594   
1 -5.54563  86.28183 -237.33537  0.0  28.02407 -0.00002 -0.00001  20.43618   

        8    9   ...       56       57        58       59   60        61   62  \
0 -0.00002  0.0  ...  3.55258 -0.00001 -36.80355 -0.00002  0.0 -39.90218  0.0   
1 -0.00002  0.0  ...  3.55258 -0.00001 -37.02589 -0.00002  0.0 -39.69159  0.0   

    63   64        65  
0  0.0  0.0  14.83583  
1  0.0  0.0  14.83426  

[2 rows x 66 columns]


In [14]:
!pip install torch scipy numpy pandas scikit-learn

In [16]:
"""
=============================================================================
PS2 — Data Loading + ST-GCN + Two-Layer Clinical Transformer Pipeline
=============================================================================
Datasets:
  UI-PRMD  : Kinect Positions (66 cols = 22 joints × 3 xyz)
             correct   → Segmented Movements/Kinect/Positions/
             incorrect → Incorrect Segmented Movements/Kinect/Positions/
             filename  : m{mov:02d}_s{subj:02d}_e{ex:02d}_positions[_inc].txt
             label     : 1 = correct, 0 = incorrect

  UCO      : dataset_3d_with_angles.json
             432 samples, per-frame l_hip/l_knee/l_ankle (+ r_* sparse)
             label     : all are correct reps (quality supervision later)

Output per sample (transformer-ready):
  keypoints : (TARGET_LEN, N_JOINTS, 3)  float32   ← ST-GCN input
  adj       : (N_JOINTS, N_JOINTS)        float32   ← fixed anatomical graph
  label     : int                                   ← 0=incorrect, 1=correct
  exercise  : str
  subject   : str                                   ← for LOSO split
  source    : str                                   ← 'uiprmd' | 'uco'

UI-PRMD Kinect joint index map (22 joints, 0-indexed):
  0  SpineBase      1  SpineMid       2  Neck           3  Head
  4  ShoulderLeft   5  ElbowLeft      6  WristLeft      7  HandLeft
  8  ShoulderRight  9  ElbowRight     10 WristRight     11 HandRight
  12 HipLeft        13 KneeLeft       14 AnkleLeft      15 FootLeft
  16 HipRight       17 KneeRight      18 AnkleRight     19 FootRight
  20 SpineShoulder  21 HandTipLeft    (some versions vary — validated below)

Lower-limb joints used (6):
  LEFT  : HipLeft(12), KneeLeft(13),  AnkleLeft(14)
  RIGHT : HipRight(16), KneeRight(17), AnkleRight(18)
"""

import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.signal import savgol_filter
from scipy.interpolate import interp1d
from typing import Dict, List, Optional, Tuple


#  config

BASE_DIR      = '/content/drive/MyDrive/datasets'
UCO_JSON      = os.path.join(BASE_DIR, 'dataset_3d_with_angles.json')
TARGET_LEN    = 150       # fixed sequence length for all samples
N_JOINTS      = 6         # lower-limb only: L_hip, L_knee, L_ankle, R_hip, R_knee, R_ankle
IN_CHANNELS   = 3         # x, y, z
SG_WIN        = 7
SG_POLY       = 2
MIN_FRAMES    = 20

# UI-PRMD: 22-joint Kinect skeleton column indices (col = joint_idx * 3 + axis)
# Lower-limb joint indices in the 22-joint map
JOINT_MAP = {
    'l_hip'  : 12,
    'l_knee' : 13,
    'l_ankle': 14,
    'r_hip'  : 16,
    'r_knee' : 17,
    'r_ankle': 18,
}
JOINT_ORDER = ['l_hip', 'l_knee', 'l_ankle', 'r_hip', 'r_knee', 'r_ankle']

# UI-PRMD movement → exercise name mapping
MOVEMENT_MAP = {
    'm01': 'deep_squat',
    'm02': 'hurdle_step',
    'm03': 'inline_lunge',
    'm04': 'side_lunge',       # lower limb relevant
    'm05': 'sit_to_stand',
    'm06': 'standing_active_straight_leg_raise',
    'm07': 'standing_shoulder_abduction',   # upper — include but low weight
    'm08': 'standing_shoulder_extension',   # upper
    'm09': 'standing_shoulder_internal_rotation',  # upper
    'm10': 'standing_shoulder_scaption',    # upper
}

# Lower-limb relevant movements (use for lower-limb model training)
LOWER_LIMB_MOVEMENTS = {'m01', 'm02', 'm03', 'm04', 'm05', 'm06'}


#  anatomical graph (ST-GCN adjacency)

def build_adjacency() -> np.ndarray:
    """
    Fixed anatomical adjacency matrix for 6 lower-limb joints.
    Joint order: [L_hip(0), L_knee(1), L_ankle(2), R_hip(3), R_knee(4), R_ankle(5)]

    Edges:
      Anatomical bones  : L_hip-L_knee, L_knee-L_ankle (left chain)
                          R_hip-R_knee, R_knee-R_ankle (right chain)
      Bilateral links   : L_hip-R_hip  (pelvis)
                          L_knee-R_knee, L_ankle-R_ankle (cross-body)
      Self-connections  : all diagonal = 1 (standard ST-GCN)
    """
    A = np.zeros((N_JOINTS, N_JOINTS), dtype=np.float32)

    edges = [
        (0, 1),   # L_hip   → L_knee
        (1, 2),   # L_knee  → L_ankle
        (3, 4),   # R_hip   → R_knee
        (4, 5),   # R_knee  → R_ankle
        (0, 3),   # L_hip   ↔ R_hip   (pelvis)
        (1, 4),   # L_knee  ↔ R_knee  (cross)
        (2, 5),   # L_ankle ↔ R_ankle (cross)
    ]

    for i, j in edges:
        A[i, j] = 1.0
        A[j, i] = 1.0

    # self-connections
    np.fill_diagonal(A, 1.0)

    # symmetric normalisation: D^{-1/2} A D^{-1/2}
    D = np.diag(A.sum(axis=1))
    D_inv_sqrt = np.diag(1.0 / np.sqrt(A.sum(axis=1) + 1e-6))
    A_norm = D_inv_sqrt @ A @ D_inv_sqrt

    return A_norm.astype(np.float32)


ADJ = build_adjacency()   # (6, 6) — shared across all samples


#  signal processing helpers

def _smooth(arr: np.ndarray) -> np.ndarray:
    if len(arr) < SG_WIN + 2:
        return arr
    win = SG_WIN if len(arr) > SG_WIN else (len(arr) if len(arr) % 2 == 1 else len(arr) - 1)
    win = max(win, SG_POLY + 2 if (SG_POLY + 2) % 2 == 1 else SG_POLY + 3)
    return savgol_filter(arr, window_length=win, polyorder=SG_POLY)


def _interpolate_to_length(arr: np.ndarray, target: int) -> np.ndarray:
    """Resample 1-D array to target length."""
    if len(arr) == target:
        return arr
    x_old = np.linspace(0, 1, len(arr))
    x_new = np.linspace(0, 1, target)
    return np.interp(x_new, x_old, arr)


def _normalise_keypoints(kps: np.ndarray) -> np.ndarray:
    """
    Centre skeleton on hip midpoint, scale by hip-to-knee distance.
    kps: (T, J, 3)
    Returns normalised (T, J, 3).
    """
    # hip midpoint = mean of L_hip(0) and R_hip(3)
    hip_mid = (kps[:, 0, :] + kps[:, 3, :]) / 2.0   # (T, 3)
    kps = kps - hip_mid[:, np.newaxis, :]             # centre

    # scale by mean L hip-to-knee distance over sequence
    bone_len = np.mean(np.linalg.norm(kps[:, 0, :] - kps[:, 1, :], axis=1)) + 1e-6
    kps = kps / bone_len

    return kps.astype(np.float32)


def _fill_missing(kps: np.ndarray) -> np.ndarray:
    """
    Linear interpolation for zero-columns (missing joints in UCO right side).
    kps: (T, J, 3)
    """
    T, J, C = kps.shape
    for j in range(J):
        for c in range(C):
            col = kps[:, j, c]
            zero_mask = (col == 0.0)
            if zero_mask.all():
                # entire joint missing — mirror from opposite side
                opp = j + 3 if j < 3 else j - 3
                kps[:, j, c] = kps[:, opp, c]
            elif zero_mask.any():
                # partial — interpolate
                valid = ~zero_mask
                if valid.sum() >= 2:
                    f = interp1d(np.where(valid)[0], col[valid],
                                 bounds_error=False, fill_value='extrapolate')
                    kps[:, j, c] = f(np.arange(T))
    return kps


def process_keypoints(raw: np.ndarray) -> np.ndarray:
    """
    Full preprocessing pipeline for (T, J, 3) keypoint array.
    1. Fill missing (zero) joints
    2. Smooth each joint-axis independently
    3. Resample to TARGET_LEN
    4. Normalise (centre + scale)
    Returns (TARGET_LEN, J, 3) float32.
    """
    T, J, C = raw.shape

    if T < MIN_FRAMES:
        raise ValueError(f"Sequence too short: {T} frames")

    # smooth
    for j in range(J):
        for c in range(C):
            raw[:, j, c] = _smooth(raw[:, j, c])

    # fill missing
    raw = _fill_missing(raw)

    # resample to TARGET_LEN
    resampled = np.zeros((TARGET_LEN, J, C), dtype=np.float32)
    for j in range(J):
        for c in range(C):
            resampled[:, j, c] = _interpolate_to_length(raw[:, j, c], TARGET_LEN)

    # normalise
    resampled = _normalise_keypoints(resampled)

    return resampled


#  UI-PRMD loader

def load_uiprmd_sample(filepath: str) -> np.ndarray:
    """
    Load one UI-PRMD Kinect positions file.
    Returns (T, 6, 3) array for the 6 lower-limb joints.
    File: T rows × 66 cols (22 joints × 3 axes, row-major: j0x j0y j0z j1x ...)
    """
    df  = pd.read_csv(filepath, header=None)
    arr = df.values.astype(np.float32)   # (T, 66)

    T   = arr.shape[0]
    kps = np.zeros((T, N_JOINTS, 3), dtype=np.float32)

    for ji, jname in enumerate(JOINT_ORDER):
        col_base = JOINT_MAP[jname] * 3
        kps[:, ji, 0] = arr[:, col_base]       # x
        kps[:, ji, 1] = arr[:, col_base + 1]   # y
        kps[:, ji, 2] = arr[:, col_base + 2]   # z

    return kps   # (T, 6, 3)


def load_uiprmd_dataset(
    lower_limb_only: bool = True,
    use_kinect: bool = True,
) -> List[dict]:
    """
    Load all UI-PRMD samples (correct + incorrect).
    Returns list of sample dicts.
    """
    modality = 'Kinect' if use_kinect else 'Vicon'
    samples  = []

    for label_name, label_val, inc_suffix in [
        ('Segmented Movements',           1, ''),
        ('Incorrect Segmented Movements', 0, '_inc'),
    ]:
        pos_dir = os.path.join(BASE_DIR, label_name, modality, 'Positions')
        if not os.path.exists(pos_dir):
            print(f"WARNING: {pos_dir} not found — skipping")
            continue

        for fname in sorted(os.listdir(pos_dir)):
            if not fname.endswith('.txt'):
                continue

            # parse filename: m01_s01_e01_positions[_inc].txt
            parts    = fname.replace('_positions', '').replace(inc_suffix, '').replace('.txt', '').split('_')
            if len(parts) < 3:
                continue
            movement = parts[0]   # m01..m10
            subject  = parts[1]   # s01..s10
            exercise = parts[2]   # e01..e10

            if lower_limb_only and movement not in LOWER_LIMB_MOVEMENTS:
                continue

            fpath = os.path.join(pos_dir, fname)
            try:
                raw = load_uiprmd_sample(fpath)
                kps = process_keypoints(raw)
            except Exception as ex:
                print(f"SKIP {fname}: {ex}")
                continue

            samples.append({
                'keypoints': kps,             # (150, 6, 3)
                'adj':       ADJ,             # (6, 6)
                'label':     label_val,       # 1=correct, 0=incorrect
                'exercise':  MOVEMENT_MAP.get(movement, movement),
                'subject':   f'uiprmd_{subject}',
                'source':    'uiprmd',
                'movement':  movement,
            })

    print(f"UI-PRMD loaded: {len(samples)} samples")
    return samples


#  UCO PhyRehab loader

UCO_JOINT_MAP = {
    'l_hip':   0,
    'l_knee':  1,
    'l_ankle': 2,
    'r_hip':   3,
    'r_knee':  4,
    'r_ankle': 5,
}

UCO_FRAME_KEYS = {
    'l_hip':   'l_hip',
    'l_knee':  'l_knee',
    'l_ankle': 'l_ankle',
    'r_hip':   'r_hip',
    'r_knee':  'r_knee',
    'r_ankle': 'r_ankle',
}


def load_uco_sample(sample: dict) -> np.ndarray:
    """
    Parse one UCO JSON sample into (T, 6, 3) keypoint array.
    Right-side joints are sparse (zero if missing — _fill_missing handles it).
    """
    frames = sample['frames']
    T      = len(frames)

    kps = np.zeros((T, N_JOINTS, 3), dtype=np.float32)

    for t, frame in enumerate(frames):
        joints = frame['joints']
        for ji, jname in enumerate(JOINT_ORDER):
            uco_key = UCO_FRAME_KEYS[jname]
            if uco_key in joints and isinstance(joints[uco_key], dict):
                jd = joints[uco_key]
                kps[t, ji, 0] = float(jd.get('x', 0.0))
                kps[t, ji, 1] = float(jd.get('y', 0.0))
                kps[t, ji, 2] = float(jd.get('z', 0.0))
            # else stays 0 — filled by _fill_missing

    return kps   # (T, 6, 3)


def load_uco_dataset() -> List[dict]:
    """
    Load all UCO PhyRehab samples.
    All UCO samples are correct reps (label=1).
    Quality score supervision added later via KIMORE fine-tuning.
    """
    with open(UCO_JSON) as f:
        data = json.load(f)['data']

    samples = []
    skipped = 0

    for i, sample in enumerate(data):
        if sample.get('body', '') != 'lower':
            continue   # skip upper-body samples

        try:
            raw = load_uco_sample(sample)
            kps = process_keypoints(raw)
        except Exception as ex:
            skipped += 1
            continue

        samples.append({
            'keypoints': kps,
            'adj':       ADJ,
            'label':     1,    # all correct reps in UCO
            'exercise':  f"uco_ex{sample.get('exercise', '00')}",
            'subject':   f"uco_folder{sample.get('folder', '0')}",
            'source':    'uco',
            'side':      sample.get('side', 'unknown'),
        })

    print(f"UCO loaded: {len(samples)} samples ({skipped} skipped)")
    return samples


#   dataset builder

def build_master_dataset(
    lower_limb_only: bool = True,
    use_uco: bool = True,
) -> List[dict]:
    """
    Combine UI-PRMD + UCO into one list.
    This is the input to the PyTorch Dataset.
    """
    samples = load_uiprmd_dataset(lower_limb_only=lower_limb_only)
    if use_uco:
        samples += load_uco_dataset()

    print(f"\nMaster dataset: {len(samples)} total samples")
    print(f"  Correct:   {sum(s['label']==1 for s in samples)}")
    print(f"  Incorrect: {sum(s['label']==0 for s in samples)}")
    print(f"  Sources:   { {s['source'] for s in samples} }")
    return samples


class RehabDataset(Dataset):
    """
    PyTorch Dataset wrapping the master sample list.
    Returns tensors ready for ST-GCN → Transformer.
    """

    def __init__(self, samples: List[dict], augment: bool = False):
        self.samples = samples
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s   = self.samples[idx]
        kps = s['keypoints'].copy()   # (150, 6, 3)

        if self.augment:
            kps = self._augment(kps)

        # ST-GCN expects (C, T, V) = (3, 150, 6)
        kps_tensor = torch.from_numpy(kps).permute(2, 0, 1).float()   # (3, 150, 6)
        adj_tensor = torch.from_numpy(s['adj']).float()                # (6, 6)
        label      = torch.tensor(s['label'], dtype=torch.long)

        return {
            'keypoints': kps_tensor,   # (3, 150, 6)
            'adj':       adj_tensor,   # (6, 6)
            'label':     label,
            'exercise':  s['exercise'],
            'subject':   s['subject'],
            'source':    s['source'],
        }

    def _augment(self, kps: np.ndarray) -> np.ndarray:
        """
        On-the-fly augmentation.
        kps: (150, 6, 3)
        """
        # 1. temporal jitter ±20%
        if np.random.rand() < 0.5:
            factor  = np.random.uniform(0.8, 1.2)
            new_len = int(TARGET_LEN * factor)
            new_len = max(new_len, MIN_FRAMES)
            stretched = np.zeros((new_len, N_JOINTS, 3), dtype=np.float32)
            for j in range(N_JOINTS):
                for c in range(3):
                    stretched[:, j, c] = _interpolate_to_length(kps[:, j, c], new_len)
            kps = np.zeros((TARGET_LEN, N_JOINTS, 3), dtype=np.float32)
            for j in range(N_JOINTS):
                for c in range(3):
                    kps[:, j, c] = _interpolate_to_length(stretched[:, j, c], TARGET_LEN)

        # 2. mirror flip (swap L↔R)
        if np.random.rand() < 0.5:
            # swap: L_hip(0)↔R_hip(3), L_knee(1)↔R_knee(4), L_ankle(2)↔R_ankle(5)
            kps[:, [0,1,2,3,4,5], :] = kps[:, [3,4,5,0,1,2], :]
            kps[:, :, 0] *= -1.0   # flip x axis

        # 3. Gaussian noise σ=2mm (positions already normalised — use 0.02)
        if np.random.rand() < 0.5:
            kps += np.random.randn(*kps.shape).astype(np.float32) * 0.02

        # 4. small rotation around vertical axis (±10°)
        if np.random.rand() < 0.3:
            theta = np.radians(np.random.uniform(-10, 10))
            cos_t, sin_t = np.cos(theta), np.sin(theta)
            x = kps[:, :, 0].copy()
            z = kps[:, :, 2].copy()
            kps[:, :, 0] = cos_t * x - sin_t * z
            kps[:, :, 2] = sin_t * x + cos_t * z

        return kps


# LOSO split

def loso_split(
    samples: List[dict],
    held_out_subject: str
) -> Tuple[List[dict], List[dict]]:
    """
    Leave-One-Subject-Out split.
    Use 'uiprmd_s01' .. 'uiprmd_s10' as held_out_subject.
    UCO samples always go to train (no subject overlap with UI-PRMD).
    """
    train = [s for s in samples if s['subject'] != held_out_subject]
    test  = [s for s in samples if s['subject'] == held_out_subject]
    print(f"LOSO held-out: {held_out_subject} → train={len(train)}, test={len(test)}")
    return train, test


def get_all_uiprmd_subjects(samples: List[dict]) -> List[str]:
    return sorted({s['subject'] for s in samples if s['source'] == 'uiprmd'})


#  scalar feature extractor

def extract_scalar_features(kps: np.ndarray) -> np.ndarray:
    """
    Compute 10 scalar features from (TARGET_LEN, 6, 3) keypoints.
    These go directly to XGBoost, bypassing ST-GCN.

    Features:
      0  L knee ROM (degrees, computed from positions)
      1  R knee ROM
      2  L knee peak angle
      3  R knee peak angle
      4  L/R ROM symmetry index (%)
      5  max angular velocity (°/s)
      6  mean jerk proxy (position jerk norm)
      7  trunk lean proxy (hip-spine deviation at peak)
      8  L/R temporal lag (frames)
      9  smoothness proxy (1 / position jerk std)
    """
    def _angle_from_positions(a, b, c):
        """Angle at joint b given positions a, b, c. Returns degrees array."""
        ba = a - b
        bc = c - b
        cos_ang = np.sum(ba * bc, axis=1) / (
            np.linalg.norm(ba, axis=1) * np.linalg.norm(bc, axis=1) + 1e-6
        )
        return np.degrees(np.arccos(np.clip(cos_ang, -1, 1)))

    # joint indices: L_hip=0, L_knee=1, L_ankle=2, R_hip=3, R_knee=4, R_ankle=5
    l_knee_ang = _angle_from_positions(kps[:, 0, :], kps[:, 1, :], kps[:, 2, :])
    r_knee_ang = _angle_from_positions(kps[:, 3, :], kps[:, 4, :], kps[:, 5, :])

    l_rom = float(np.max(l_knee_ang) - np.min(l_knee_ang))
    r_rom = float(np.max(r_knee_ang) - np.min(r_knee_ang))
    l_peak = float(np.min(l_knee_ang))
    r_peak = float(np.min(r_knee_ang))
    sym   = float(abs(l_rom - r_rom) / ((l_rom + r_rom) / 2.0 + 1e-6) * 100.0)

    # angular velocity proxy (mean frame-to-frame angle change)
    l_vel    = np.abs(np.diff(l_knee_ang)) * 30.0   # FPS=30
    max_vel  = float(np.max(l_vel))

    # jerk proxy (norm of 2nd derivative of hip position)
    hip_mid  = (kps[:, 0, :] + kps[:, 3, :]) / 2.0
    jerk_vec = np.diff(np.diff(hip_mid, axis=0), axis=0)
    jerk_norm= np.linalg.norm(jerk_vec, axis=1)
    mean_jerk= float(np.mean(jerk_norm))

    # trunk lean proxy: hip midpoint x-deviation at peak knee bend
    peak_idx    = int(np.argmin(l_knee_ang))
    trunk_lean  = float(abs(hip_mid[peak_idx, 0] - hip_mid[0, 0]))

    # temporal lag between L and R knee angle curves
    corr  = np.correlate(l_knee_ang - l_knee_ang.mean(),
                         r_knee_ang - r_knee_ang.mean(), mode='full')
    lag   = float(abs(np.argmax(corr) - (len(l_knee_ang) - 1)))

    smoothness = float(1.0 / (np.std(jerk_norm) + 1e-6))

    return np.array([
        l_rom / 180.0,
        r_rom / 180.0,
        l_peak / 180.0,
        r_peak / 180.0,
        sym / 100.0,
        max_vel / 200.0,
        mean_jerk / 10.0,
        trunk_lean,
        lag / TARGET_LEN,
        np.clip(smoothness / 100.0, 0, 1),
    ], dtype=np.float32)


#  ST-GCN

class STGCNBlock(nn.Module):
    """
    One ST-GCN block: spatial GCN + temporal conv.
    Input : (B, C_in, T, V)
    Output: (B, C_out, T, V)
    """

    def __init__(self, c_in: int, c_out: int, kernel_size: int = 9, stride: int = 1):
        super().__init__()
        pad = (kernel_size - 1) // 2

        self.gcn = nn.Linear(c_in, c_out)   # spatial (applied per node)
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(c_out),
            nn.ReLU(),
            nn.Conv2d(c_out, c_out, kernel_size=(kernel_size, 1),
                      stride=(stride, 1), padding=(pad, 0)),
            nn.BatchNorm2d(c_out),
            nn.Dropout(0.1),
        )
        self.residual = (
            nn.Sequential(
                nn.Conv2d(c_in, c_out, kernel_size=1, stride=(stride, 1)),
                nn.BatchNorm2d(c_out),
            ) if c_in != c_out or stride != 1 else nn.Identity()
        )
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
        # x: (B, C, T, V),  A: (V, V) or (B, V, V)
        B, C, T, V = x.shape

        # ensure A is always (V, V) — take first element if batched
        if A.dim() == 3:
            A = A[0]   # (V, V)

        # spatial GCN: aggregate neighbour features via A
        # reshape to (B*T, V, C), multiply by A, apply linear
        x_s = x.permute(0, 2, 3, 1).contiguous().view(B * T, V, C)   # (B*T, V, C)
        x_s = torch.bmm(A.unsqueeze(0).expand(B * T, -1, -1), x_s)   # (B*T, V, C)
        x_s = self.gcn(x_s)                                            # (B*T, V, C_out)
        x_s = x_s.view(B, T, V, -1).permute(0, 3, 1, 2)               # (B, C_out, T, V)

        # temporal conv
        out = self.tcn(x_s) + self.residual(x)
        return self.relu(out)


class STGCN(nn.Module):
    """
    3-block ST-GCN spatial encoder.
    Input : (B, 3, T, 6)    — (batch, xyz, frames, joints)
    Output: (B, T, 128)     — frame-level features for Transformer
    """

    def __init__(self):
        super().__init__()
        self.blocks = nn.ModuleList([
            STGCNBlock(3,   32,  kernel_size=9),
            STGCNBlock(32,  64,  kernel_size=9),
            STGCNBlock(64, 128,  kernel_size=9),
        ])
        self.bn_input = nn.BatchNorm1d(3 * 6)   # normalise input channels×joints

    def forward(self, x: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
        # x: (B, 3, T, 6),  A: (6, 6)
        B, C, T, V = x.shape

        # input BN
        x_flat = x.permute(0, 2, 1, 3).contiguous().view(B * T, C * V)
        x_flat = self.bn_input(x_flat)
        x = x_flat.view(B, T, C, V).permute(0, 2, 1, 3)   # back to (B, C, T, V)

        for block in self.blocks:
            x = block(x, A)   # (B, 128, T, 6)

        # pool over joints → frame-level features
        x = x.mean(dim=3)          # (B, 128, T)
        x = x.permute(0, 2, 1)    # (B, T, 128)
        return x


#  Two-Layer Clinical Transformer

class ClinicalTransformer(nn.Module):
    """
    Two-layer transformer with explicit clinical interpretation per layer.

    Layer 1 — Temporal correlation:
      Self-attention across T=150 frames.
      Learns which frames are related (e.g. frame 45 ↔ frame 120 = both peaks).
      Input: frame-level ST-GCN features (B, T, 128).

    Layer 2 — Clinical interpretation:
      Self-attention over the temporally-aware representations.
      Learns what the temporal correlations mean clinically
      (e.g. 'peak valgus coincides with peak velocity = dangerous pattern').
      CLS token aggregates the full rep into a single vector.

    Output: (B, 128) rep-level summary vector → XGBoost / quality head.
    """

    def __init__(
        self,
        d_model: int = 128,
        n_heads: int = 4,
        d_ff: int = 256,
        dropout: float = 0.1,
    ):
        super().__init__()

        # positional encoding
        self.pos_enc = self._build_pos_enc(TARGET_LEN + 1, d_model)  # +1 for CLS

        # CLS token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

        # Layer 1: temporal correlation
        self.layer1 = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True,
            norm_first=True,   # pre-norm for stability
        )
        self.norm1 = nn.LayerNorm(d_model)

        # Layer 2: clinical interpretation
        self.layer2 = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True,
            norm_first=True,
        )
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def _build_pos_enc(max_len: int, d_model: int) -> nn.Parameter:
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() *
                        (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        return nn.Parameter(pe.unsqueeze(0), requires_grad=False)  # (1, max_len, d_model)

    def forward(
        self,
        x: torch.Tensor,
        return_attention: bool = False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        x: (B, T, 128) ST-GCN frame features
        Returns: (B, 128) CLS vector, optionally layer-2 attention weights
        """
        B, T, D = x.shape

        # prepend CLS token
        cls  = self.cls_token.expand(B, -1, -1)        # (B, 1, 128)
        x    = torch.cat([cls, x], dim=1)               # (B, T+1, 128)
        x    = x + self.pos_enc[:, :T + 1, :]
        x    = self.dropout(x)

        # Layer 1: temporal correlation
        x    = self.layer1(x)
        x    = self.norm1(x)

        # Layer 2: clinical interpretation
        # hook attention weights for therapist dashboard heatmap
        attn_weights = None
        if return_attention:
            # manually run attention to extract weights
            x2, attn_weights = self._layer2_with_attn(x)
            x = self.norm2(x2)
        else:
            x = self.layer2(x)
            x = self.norm2(x)

        # CLS token = rep summary
        cls_out = x[:, 0, :]   # (B, 128)
        return cls_out, attn_weights

    def _layer2_with_attn(
        self, x: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Run layer2 and return (output, attention_weights)."""
        # access layer2 self-attention directly
        sa  = self.layer2.self_attn
        x_n = self.layer2.norm1(x) if self.layer2.norm_first else x
        attn_out, attn_w = sa(x_n, x_n, x_n, need_weights=True, average_attn_weights=True)
        # continue through the rest of the encoder layer manually
        x2  = x + self.layer2.dropout1(attn_out)
        x_n2 = self.layer2.norm2(x2) if self.layer2.norm_first else x2
        ff_out = self.layer2.linear2(
            self.layer2.dropout(self.layer2.activation(self.layer2.linear1(x_n2)))
        )
        x2 = x2 + self.layer2.dropout2(ff_out)
        return x2, attn_w   # attn_w: (B, T+1, T+1)


# full model

class RehabNet(nn.Module):
    """
    Full pipeline: ST-GCN → Two-Layer Clinical Transformer → Classification head.
    For XGBoost head: call encode() to get (B, 128) embeddings.
    """

    def __init__(self, n_classes: int = 2):
        super().__init__()
        self.stgcn       = STGCN()
        self.transformer = ClinicalTransformer()
        self.classifier  = nn.Sequential(
            nn.Linear(128 + 10, 64),   # 128 CLS + 10 scalar features
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, n_classes),
        )

    def encode(
        self,
        keypoints: torch.Tensor,
        adj: torch.Tensor,
        return_attention: bool = False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        Returns (B, 128) rep embedding + optional attention weights.
        adj can be (6,6) or (B,6,6) — reduced to (6,6) inside STGCNBlock.
        """
        if adj.dim() == 3:
            adj = adj[0]   # (6, 6)
        frame_feats = self.stgcn(keypoints, adj)
        cls_vec, attn = self.transformer(frame_feats, return_attention)
        return cls_vec, attn

    def forward(
        self,
        keypoints: torch.Tensor,
        adj: torch.Tensor,
        scalars: torch.Tensor,
    ) -> torch.Tensor:
        """
        keypoints : (B, 3, T, 6)
        adj       : (B, 6, 6) or (6, 6)
        scalars   : (B, 10)
        Returns   : (B, n_classes) logits
        """
        cls_vec, _ = self.encode(keypoints, adj)
        combined   = torch.cat([cls_vec, scalars], dim=1)
        return self.classifier(combined)


#  DataLoader builder

def build_dataloaders(
    train_samples: List[dict],
    test_samples:  List[dict],
    batch_size: int = 32,
) -> Tuple[DataLoader, DataLoader]:
    """Build train/test DataLoaders with scalar features attached."""

    # attach scalar features to each sample
    for s in train_samples + test_samples:
        s['scalars'] = extract_scalar_features(s['keypoints'])

    train_ds = RehabDataset(train_samples, augment=True)
    test_ds  = RehabDataset(test_samples,  augment=False)

    # custom collate to handle scalar features
    def collate(batch):
        return {
            'keypoints': torch.stack([b['keypoints'] for b in batch]),
            'adj':       torch.stack([b['adj']       for b in batch]),
            'label':     torch.stack([b['label']     for b in batch]),
            'scalars':   torch.tensor(
                             np.stack([b['scalars'] for b in batch]),
                             dtype=torch.float32
                         ) if 'scalars' in batch[0] else None,
            'exercise':  [b['exercise'] for b in batch],
            'subject':   [b['subject']  for b in batch],
        }

    train_dl = DataLoader(train_ds, batch_size=batch_size,
                          shuffle=True,  collate_fn=collate, num_workers=2)
    test_dl  = DataLoader(test_ds,  batch_size=batch_size,
                          shuffle=False, collate_fn=collate, num_workers=2)
    return train_dl, test_dl


#  training loop
def train_one_epoch(
    model: RehabNet,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> float:
    model.train()
    total_loss = 0.0
    adj_shared = torch.from_numpy(ADJ).to(device)

    for batch in loader:
        kps     = batch['keypoints'].to(device)   # (B, 3, 150, 6)
        labels  = batch['label'].to(device)
        scalars = batch['scalars'].to(device) if batch['scalars'] is not None \
                  else torch.zeros(kps.shape[0], 10, device=device)

        logits = model(kps, adj_shared, scalars)
        loss   = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(
    model: RehabNet,
    loader: DataLoader,
    device: torch.device,
) -> dict:
    model.eval()
    adj_shared = torch.from_numpy(ADJ).to(device)
    all_preds, all_labels = [], []

    for batch in loader:
        kps     = batch['keypoints'].to(device)
        labels  = batch['label'].to(device)
        scalars = batch['scalars'].to(device) if batch['scalars'] is not None \
                  else torch.zeros(kps.shape[0], 10, device=device)

        logits = model(kps, adj_shared, scalars)
        preds  = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    from sklearn.metrics import accuracy_score, f1_score, classification_report
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro')
    print(classification_report(all_labels, all_preds,
                                 target_names=['incorrect', 'correct']))
    return {'accuracy': acc, 'f1_macro': f1}


#  LOSO training

def run_loso_training(
    n_epochs: int = 30,
    batch_size: int = 32,
    lr: float = 1e-3,
    device_str: str = 'cuda' if torch.cuda.is_available() else 'cpu',
):
    """
    Full LOSO cross-validation run.
    Trains one model per UI-PRMD subject, reports mean ± std accuracy.
    """
    device  = torch.device(device_str)
    samples = build_master_dataset(lower_limb_only=True, use_uco=True)
    subjects = get_all_uiprmd_subjects(samples)

    results = []
    for subj in subjects:
        print(f"\n{'='*50}")
        print(f"LOSO fold: held-out = {subj}")
        train_s, test_s = loso_split(samples, subj)
        train_dl, test_dl = build_dataloaders(train_s, test_s, batch_size)

        model     = RehabNet(n_classes=2).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

        for epoch in range(n_epochs):
            loss = train_one_epoch(model, train_dl, optimizer, device)
            scheduler.step()
            if (epoch + 1) % 10 == 0:
                print(f"  Epoch {epoch+1}/{n_epochs}  loss={loss:.4f}")

        metrics = evaluate(model, test_dl, device)
        results.append(metrics)
        print(f"  → acc={metrics['accuracy']:.3f}  f1={metrics['f1_macro']:.3f}")

    accs = [r['accuracy'] for r in results]
    print(f"\nLOSO summary: acc = {np.mean(accs):.3f} ± {np.std(accs):.3f}")
    return results



if __name__ == '__main__':
    print("Architecture smoke test")
    device = torch.device('cpu')
    B = 4

    kps     = torch.randn(B, 3, TARGET_LEN, N_JOINTS)
    adj     = torch.from_numpy(ADJ).unsqueeze(0).expand(B, -1, -1)
    scalars = torch.randn(B, 10)

    model  = RehabNet(n_classes=2)
    logits = model(kps, adj, scalars)
    print(f"  Input  kps:     {kps.shape}")
    print(f"  Input  adj:     {adj.shape}")
    print(f"  Input  scalars: {scalars.shape}")
    print(f"  Output logits:  {logits.shape}")

    # test attention extraction
    cls_vec, attn = model.encode(kps, adj, return_attention=True)
    print(f"  CLS embedding:  {cls_vec.shape}")
    print(f"  Attn weights:   {attn.shape if attn is not None else None}")

    # adjacency matrix
    print(f"\nAdjacency matrix (normalised)")
    print(np.round(ADJ, 3))

    print("\n Smoke test passed")

── Architecture smoke test (no data needed) ────────────────
  Input  kps:     torch.Size([4, 3, 150, 6])
  Input  adj:     torch.Size([4, 6, 6])
  Input  scalars: torch.Size([4, 10])
  Output logits:  torch.Size([4, 2])
  CLS embedding:  torch.Size([4, 128])
  Attn weights:   torch.Size([4, 151, 151])

── Adjacency matrix (normalised) ───────────────────────────
[[0.333 0.289 0.    0.333 0.    0.   ]
 [0.289 0.25  0.289 0.    0.25  0.   ]
 [0.    0.289 0.333 0.    0.    0.333]
 [0.333 0.    0.    0.333 0.289 0.   ]
 [0.    0.25  0.    0.289 0.25  0.289]
 [0.    0.    0.333 0.    0.289 0.333]]

── Smoke test passed ───────────────────────────────────────

To load data and run LOSO, call:
  results = run_loso_training(n_epochs=30, batch_size=32)


In [17]:
# Cell 4 — verify joint mapping FIRST before anything else
import pandas as pd
df = pd.read_csv(
    '/content/drive/MyDrive/datasets/Segmented Movements/Kinect/Positions/m01_s01_e01_positions.txt',
    header=None
)
# joint 12 = HipLeft → cols 36,37,38
# joint 13 = KneeLeft → cols 39,40,41
# joint 14 = AnkleLeft → cols 42,43,44
print("HipLeft (should be near body centre):")
print(df.iloc[:3, 36:39].values)
print("KneeLeft (should be below hip):")
print(df.iloc[:3, 39:42].values)
print("AnkleLeft (should be lowest):")
print(df.iloc[:3, 42:45].values)

HipLeft (should be near body centre):
[[-2.302860e+01 -1.000000e-05  0.000000e+00]
 [-2.296607e+01 -1.000000e-05  0.000000e+00]
 [-2.291306e+01 -1.000000e-05  0.000000e+00]]
KneeLeft (should be below hip):
[[-1.929371e+01  1.000000e-05 -4.000000e-05]
 [-1.929776e+01  1.000000e-05 -4.000000e-05]
 [-1.930017e+01  1.000000e-05 -4.000000e-05]]
AnkleLeft (should be lowest):
[[6.88991 0.03708 3.57008]
 [6.88696 0.04082 3.57008]
 [6.88705 0.05412 3.57008]]


In [ ]:
# Cell 5 — load datasets (only run after joint mapping confirmed)
from ps2_pipeline import build_master_dataset
samples = build_master_dataset(lower_limb_only=True, use_uco=True)

In [18]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    '/content/drive/MyDrive/datasets/Segmented Movements/Kinect/Positions/m01_s01_e01_positions.txt',
    header=None
).values.astype(np.float32)

print("Shape:", df.shape)  # should be (T, 66)

# print all 22 joints — show mean position across frames
# joint i = cols i*3, i*3+1, i*3+2
print("\nJoint index | mean_x    mean_y    mean_z")
print("-" * 45)
for i in range(22):
    c = i * 3
    mx, my, mz = df[:, c].mean(), df[:, c+1].mean(), df[:, c+2].mean()
    print(f"  Joint {i:02d}  | {mx:9.3f}  {my:9.3f}  {mz:9.3f}")

Shape: (73, 66)

Joint index | mean_x    mean_y    mean_z
---------------------------------------------
  Joint 00  |    -2.679     62.243   -248.571
  Joint 01  |    -0.000     27.170      0.000
  Joint 02  |     0.000     20.203      0.000
  Joint 03  |     0.000      0.677     -0.000
  Joint 04  |     0.000      6.092      0.000
  Joint 05  |     0.000     14.975     -0.000
  Joint 06  |     1.301      0.316      0.176
  Joint 07  |    12.376      0.000      0.000
  Joint 08  |    25.891     -0.000      0.000
  Joint 09  |    26.063      0.000      0.000
  Joint 10  |    -1.734      0.220      0.176
  Joint 11  |   -15.936      0.000      0.000
  Joint 12  |   -24.004     -0.000      0.000
  Joint 13  |   -23.894      0.000     -0.000
  Joint 14  |     6.436      1.695      3.114
  Joint 15  |     0.000    -35.285     -0.000
  Joint 16  |     0.000    -38.514      0.000
  Joint 17  |     0.000      0.000     13.034
  Joint 18  |    -6.045      1.205      3.114
  Joint 19  |     0.00

In [19]:
# check vicon positions
df_v = pd.read_csv(
    '/content/drive/MyDrive/datasets/Segmented Movements/Vicon/Positions/m01_s01_e01_positions.txt',
    header=None
).values.astype(np.float32)

print("Vicon Positions shape:", df_v.shape)
print("\nJoint index | mean_x    mean_y    mean_z")
print("-" * 45)
for i in range(min(39, df_v.shape[1]//3)):
    c = i * 3
    mx, my, mz = df_v[:, c].mean(), df_v[:, c+1].mean(), df_v[:, c+2].mean()
    print(f"  Joint {i:02d}  | {mx:9.3f}  {my:9.3f}  {mz:9.3f}")

Vicon Positions shape: (191, 117)

Joint index | mean_x    mean_y    mean_z
---------------------------------------------
  Joint 00  |   -69.614     94.712   1238.841
  Joint 01  |   -64.864    -24.747   1236.324
  Joint 02  |  -223.830     97.312   1270.703
  Joint 03  |  -216.519    -33.711   1263.394
  Joint 04  |  -317.496     24.019   1147.958
  Joint 05  |  -436.875     30.054    942.350
  Joint 06  |  -205.386     31.273   1051.988
  Joint 07  |  -210.999     25.656    819.755
  Joint 08  |  -368.136    -88.217   1095.050
  Joint 09  |  -288.367    120.838   1151.154
  Joint 10  |  -314.850    251.652   1169.278
  Joint 11  |  -252.970    360.106   1260.281
  Joint 12  |  -219.120    427.686   1349.640
  Joint 13  |  -143.430    444.157   1469.902
  Joint 14  |  -160.731    505.102   1446.741
  Joint 15  |  -203.115    480.369   1516.694
  Joint 16  |  -273.005    -67.034   1142.831
  Joint 17  |  -281.087   -199.137   1157.995
  Joint 18  |  -194.367   -300.591   1243.884
  Jo

In [20]:
# also check kinect ANGLES (not positions) — might be more useful
df_ka = pd.read_csv(
    '/content/drive/MyDrive/datasets/Segmented Movements/Kinect/Angles/m01_s01_e01_angles.txt',
    header=None
).values.astype(np.float32)

print("Kinect Angles shape:", df_ka.shape)
print("First row:", df_ka[0, :12])  # first 4 joints × 3 angles

Kinect Angles shape: (73, 66)
First row: [-3.31472e+00 -3.79866e+00 -2.31144e+00  1.79215e+00 -2.21450e-01
 -2.14530e-01  2.43673e+00 -3.36650e-01 -1.45900e-01  5.00000e-05
  0.00000e+00  0.00000e+00]


In [21]:
df_v = pd.read_csv(
    '/content/drive/MyDrive/datasets/Segmented Movements/Vicon/Positions/m01_s01_e01_positions.txt',
    header=None
).values.astype(np.float32)

# UI-PRMD Vicon documented joint names (39 joints)
vicon_joint_names = [
    'RFHD','LFHD','RBHD','LBHD',           # 0-3  head
    'C7','T10','CLAV','STRN',               # 4-7  spine
    'RBAK',                                 # 8    back
    'LSHO','LUPA','LELB','LFRA','LWRA',     # 9-13 left arm
    'LWRB','LFIN',                          # 14-15
    'RSHO','RUPA','RELB','RFRA','RWRA',     # 16-20 right arm
    'RWRB','RFIN',                          # 21-22
    'LASI','RASI','LPSI','RPSI',            # 23-26 pelvis ← hips here
    'LTHI','LKNE','LTIB','LANK','LHEE',    # 27-31 left leg
    'LTOE',                                 # 32
    'RTHI','RKNE','RTIB','RANK','RHEE',    # 33-37 right leg
    'RTOE',                                 # 38
]

print("Lower-limb candidates:")
for i, name in enumerate(vicon_joint_names):
    c = i * 3
    mx, my, mz = df_v[:, c].mean(), df_v[:, c+1].mean(), df_v[:, c+2].mean()
    if any(x in name for x in ['ASI','PSI','THI','KNE','ANK','HEE','TOE']):
        print(f"  Joint {i:02d} {name:6s} | x={mx:8.1f}  y={my:8.1f}  z={mz:8.1f}")

Lower-limb candidates:
  Joint 23 LASI   | x=  -366.5  y=   150.1  z=   693.5
  Joint 24 RASI   | x=  -365.1  y=  -106.2  z=   686.3
  Joint 25 LPSI   | x=  -558.4  y=    54.8  z=   756.6
  Joint 26 RPSI   | x=  -552.5  y=   -30.4  z=   757.7
  Joint 27 LTHI   | x=  -420.7  y=   263.7  z=   575.7
  Joint 28 LKNE   | x=  -277.7  y=   325.1  z=   474.4
  Joint 30 LANK   | x=  -404.5  y=   347.2  z=    87.1
  Joint 31 LHEE   | x=  -424.3  y=   284.3  z=    65.3
  Joint 32 LTOE   | x=  -264.3  y=   391.3  z=    49.5
  Joint 33 RTHI   | x=  -403.9  y=  -233.6  z=   580.5
  Joint 34 RKNE   | x=  -273.8  y=  -296.5  z=   464.9
  Joint 36 RANK   | x=  -411.2  y=  -301.5  z=    91.4
  Joint 37 RHEE   | x=  -431.7  y=  -243.9  z=    66.1
  Joint 38 RTOE   | x=  -270.5  y=  -358.4  z=    50.5


In [22]:
samples = build_master_dataset(lower_limb_only=True, use_uco=True)

UI-PRMD loaded: 1076 samples
UCO loaded: 212 samples (4 skipped)

Master dataset: 1288 total samples
  Correct:   812
  Incorrect: 476
  Sources:   {'uiprmd', 'uco'}


In [23]:
# verify one sample has real values (not zeros)
s = samples[0]
kps = s['keypoints']  # (150, 6, 3)
print("shape:", kps.shape)
print("l_hip  z (should be ~0.5-1.0 after normalisation):", kps[:3, 0, 2])
print("l_knee z (should be < l_hip z):", kps[:3, 1, 2])
print("l_ankle z (should be lowest):", kps[:3, 2, 2])
print("any NaN:", np.isnan(kps).any())
print("any all-zero joint:", [(kps[:, j, :] == 0).all() for j in range(6)])


shape: (150, 6, 3)
l_hip  z (should be ~0.5-1.0 after normalisation): [0. 0. 0.]
l_knee z (should be < l_hip z): [-2.9065894e-05 -2.9065894e-05 -2.9065894e-05]
l_ankle z (should be lowest): [2.5941892 2.5941892 2.5941892]
any NaN: False
any all-zero joint: [np.False_, np.False_, np.False_, np.False_, np.False_, np.False_]


In [24]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    '/content/drive/MyDrive/datasets/Segmented Movements/Vicon/Positions/m01_s01_e01_positions.txt',
    header=None
).values.astype(np.float32)

# raw values for our 6 joints before any processing
joints = {'l_hip':23, 'l_knee':28, 'l_ankle':30, 'r_hip':24, 'r_knee':34, 'r_ankle':36}
print("Raw values (first 3 frames):")
for name, idx in joints.items():
    c = idx * 3
    print(f"  {name:8s}: x={df[:3,c]}  y={df[:3,c+1]}  z={df[:3,c+2]}")

print("\nVariance per axis:")
for name, idx in joints.items():
    c = idx * 3
    print(f"  {name:8s}: var_x={df[:,c].var():.1f}  var_y={df[:,c+1].var():.1f}  var_z={df[:,c+2].var():.1f}")

Raw values (first 3 frames):
  l_hip   : x=[-332.165 -335.614 -338.993]  y=[153.273 152.896 152.517]  z=[928.855 926.113 922.994]
  l_knee  : x=[-360.626 -359.66  -358.513]  y=[284.349 287.437 290.533]  z=[524.453 524.792 524.987]
  l_ankle : x=[-415.092 -415.153 -415.184]  y=[351.895 351.986 352.072]  z=[86.0083 85.9305 85.854 ]
  r_hip   : x=[-311.478 -314.567 -317.674]  y=[-108.311 -108.402 -108.502]  z=[932.19  929.04  925.542]
  r_knee  : x=[-337.379 -335.822 -334.23 ]  y=[-257.511 -260.384 -263.344]  z=[513.313 513.12  512.818]
  r_ankle : x=[-423.69  -423.597 -423.488]  y=[-306.441 -306.663 -306.876]  z=[89.7292 89.5804 89.4384]

Variance per axis:
  l_hip   : var_x=3517.5  var_y=39.7  var_z=36619.8
  l_knee  : var_x=2437.2  var_y=919.0  var_z=1472.4
  l_ankle : var_x=59.7  var_y=33.9  var_z=3.5
  r_hip   : var_x=3731.8  var_y=25.6  var_z=41664.4
  r_knee  : var_x=3032.3  var_y=1486.7  var_z=1468.0
  r_ankle : var_x=108.2  var_y=43.0  var_z=4.7


In [26]:
"""
=============================================================================
PS2 — Data Loading + ST-GCN + Two-Layer Clinical Transformer Pipeline
=============================================================================
Datasets:
  UI-PRMD  : Kinect Positions (66 cols = 22 joints × 3 xyz)
             correct   → Segmented Movements/Kinect/Positions/
             incorrect → Incorrect Segmented Movements/Kinect/Positions/
             filename  : m{mov:02d}_s{subj:02d}_e{ex:02d}_positions[_inc].txt
             label     : 1 = correct, 0 = incorrect

  UCO      : dataset_3d_with_angles.json
             432 samples, per-frame l_hip/l_knee/l_ankle (+ r_* sparse)
             label     : all are correct reps (quality supervision later)

Output per sample (transformer-ready):
  keypoints : (TARGET_LEN, N_JOINTS, 3)  float32   ← ST-GCN input
  adj       : (N_JOINTS, N_JOINTS)        float32   ← fixed anatomical graph
  label     : int                                   ← 0=incorrect, 1=correct
  exercise  : str
  subject   : str                                   ← for LOSO split
  source    : str                                   ← 'uiprmd' | 'uco'

UI-PRMD Kinect joint index map (22 joints, 0-indexed):
  0  SpineBase      1  SpineMid       2  Neck           3  Head
  4  ShoulderLeft   5  ElbowLeft      6  WristLeft      7  HandLeft
  8  ShoulderRight  9  ElbowRight     10 WristRight     11 HandRight
  12 HipLeft        13 KneeLeft       14 AnkleLeft      15 FootLeft
  16 HipRight       17 KneeRight      18 AnkleRight     19 FootRight
  20 SpineShoulder  21 HandTipLeft    (some versions vary — validated below)

Lower-limb joints used (6):
  LEFT  : HipLeft(12), KneeLeft(13),  AnkleLeft(14)
  RIGHT : HipRight(16), KneeRight(17), AnkleRight(18)
"""

import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.signal import savgol_filter
from scipy.interpolate import interp1d
from typing import Dict, List, Optional, Tuple


# ─── config ──────────────────────────────────────────────────────────────────

BASE_DIR      = '/content/drive/MyDrive/datasets'
UCO_JSON      = os.path.join(BASE_DIR, 'dataset_3d_with_angles.json')
TARGET_LEN    = 150       # fixed sequence length for all samples
N_JOINTS      = 6         # lower-limb only: L_hip, L_knee, L_ankle, R_hip, R_knee, R_ankle
IN_CHANNELS   = 3         # x, y, z
SG_WIN        = 7
SG_POLY       = 2
MIN_FRAMES    = 20

# UI-PRMD: Vicon Positions (39 joints x 3 = 117 cols)
# Confirmed from data inspection (z = vertical axis):
#   LASI(23) z=693  LKNE(28) z=474  LANK(30) z=87   ← left chain
#   RASI(24) z=686  RKNE(34) z=465  RANK(36) z=91   ← right chain
# Using LASI/RASI as hip proxies (anterior superior iliac spine)
JOINT_MAP = {
    'l_hip'  : 23,   # LASI — left anterior superior iliac spine
    'l_knee' : 28,   # LKNE — left knee
    'l_ankle': 30,   # LANK — left ankle
    'r_hip'  : 24,   # RASI — right anterior superior iliac spine
    'r_knee' : 34,   # RKNE — right knee
    'r_ankle': 36,   # RANK — right ankle
}
JOINT_ORDER = ['l_hip', 'l_knee', 'l_ankle', 'r_hip', 'r_knee', 'r_ankle']

# UI-PRMD movement → exercise name mapping
MOVEMENT_MAP = {
    'm01': 'deep_squat',
    'm02': 'hurdle_step',
    'm03': 'inline_lunge',
    'm04': 'side_lunge',       # lower limb relevant
    'm05': 'sit_to_stand',
    'm06': 'standing_active_straight_leg_raise',
    'm07': 'standing_shoulder_abduction',   # upper — include but low weight
    'm08': 'standing_shoulder_extension',   # upper
    'm09': 'standing_shoulder_internal_rotation',  # upper
    'm10': 'standing_shoulder_scaption',    # upper
}

# Lower-limb relevant movements (use for lower-limb model training)
LOWER_LIMB_MOVEMENTS = {'m01', 'm02', 'm03', 'm04', 'm05', 'm06'}


# ─── anatomical graph (ST-GCN adjacency) ─────────────────────────────────────

def build_adjacency() -> np.ndarray:
    """
    Fixed anatomical adjacency matrix for 6 lower-limb joints.
    Joint order: [L_hip(0), L_knee(1), L_ankle(2), R_hip(3), R_knee(4), R_ankle(5)]

    Edges:
      Anatomical bones  : L_hip-L_knee, L_knee-L_ankle (left chain)
                          R_hip-R_knee, R_knee-R_ankle (right chain)
      Bilateral links   : L_hip-R_hip  (pelvis)
                          L_knee-R_knee, L_ankle-R_ankle (cross-body)
      Self-connections  : all diagonal = 1 (standard ST-GCN)
    """
    A = np.zeros((N_JOINTS, N_JOINTS), dtype=np.float32)

    edges = [
        (0, 1),   # L_hip   → L_knee
        (1, 2),   # L_knee  → L_ankle
        (3, 4),   # R_hip   → R_knee
        (4, 5),   # R_knee  → R_ankle
        (0, 3),   # L_hip   ↔ R_hip   (pelvis)
        (1, 4),   # L_knee  ↔ R_knee  (cross)
        (2, 5),   # L_ankle ↔ R_ankle (cross)
    ]

    for i, j in edges:
        A[i, j] = 1.0
        A[j, i] = 1.0

    # self-connections
    np.fill_diagonal(A, 1.0)

    # symmetric normalisation: D^{-1/2} A D^{-1/2}
    D = np.diag(A.sum(axis=1))
    D_inv_sqrt = np.diag(1.0 / np.sqrt(A.sum(axis=1) + 1e-6))
    A_norm = D_inv_sqrt @ A @ D_inv_sqrt

    return A_norm.astype(np.float32)


ADJ = build_adjacency()   # (6, 6) — shared across all samples


# ─── signal processing helpers ───────────────────────────────────────────────

def _smooth(arr: np.ndarray) -> np.ndarray:
    if len(arr) < SG_WIN + 2:
        return arr
    win = SG_WIN if len(arr) > SG_WIN else (len(arr) if len(arr) % 2 == 1 else len(arr) - 1)
    win = max(win, SG_POLY + 2 if (SG_POLY + 2) % 2 == 1 else SG_POLY + 3)
    return savgol_filter(arr, window_length=win, polyorder=SG_POLY)


def _interpolate_to_length(arr: np.ndarray, target: int) -> np.ndarray:
    """Resample 1-D array to target length."""
    if len(arr) == target:
        return arr
    x_old = np.linspace(0, 1, len(arr))
    x_new = np.linspace(0, 1, target)
    return np.interp(x_new, x_old, arr)


def _normalise_keypoints(kps: np.ndarray) -> np.ndarray:
    """
    Normalise skeleton keypoints:
      1. Centre on mean hip midpoint (single reference point across all frames)
         — removes subject position in room, keeps inter-frame motion intact
      2. Scale by mean hip-to-ankle distance (full leg length)
         — removes subject height differences, preserves joint ratios

    kps: (T, J, 3)  joints: L_hip=0, L_knee=1, L_ankle=2, R_hip=3, R_knee=4, R_ankle=5
    Returns normalised (T, J, 3) where:
      - hip midpoint at origin on average
      - full leg length ~1.0
      - joints still move freely across frames
    """
    # use MEAN hip midpoint as single anchor (not per-frame — preserves motion)
    hip_mid_mean = ((kps[:, 0, :] + kps[:, 3, :]) / 2.0).mean(axis=0)  # (3,)
    kps = kps - hip_mid_mean[np.newaxis, np.newaxis, :]                  # (T, J, 3)

    # scale by mean hip-to-ankle distance (full leg length — more stable than hip-knee)
    l_leg = np.mean(np.linalg.norm(kps[:, 0, :] - kps[:, 2, :], axis=1))
    r_leg = np.mean(np.linalg.norm(kps[:, 3, :] - kps[:, 5, :], axis=1))
    leg_len = (l_leg + r_leg) / 2.0 + 1e-6
    kps = kps / leg_len

    return kps.astype(np.float32)


def _fill_missing(kps: np.ndarray) -> np.ndarray:
    """
    Linear interpolation for zero-columns (missing joints in UCO right side).
    kps: (T, J, 3)
    """
    T, J, C = kps.shape
    for j in range(J):
        for c in range(C):
            col = kps[:, j, c]
            zero_mask = (col == 0.0)
            if zero_mask.all():
                # entire joint missing — mirror from opposite side
                opp = j + 3 if j < 3 else j - 3
                kps[:, j, c] = kps[:, opp, c]
            elif zero_mask.any():
                # partial — interpolate
                valid = ~zero_mask
                if valid.sum() >= 2:
                    f = interp1d(np.where(valid)[0], col[valid],
                                 bounds_error=False, fill_value='extrapolate')
                    kps[:, j, c] = f(np.arange(T))
    return kps


def process_keypoints(raw: np.ndarray) -> np.ndarray:
    """
    Full preprocessing pipeline for (T, J, 3) keypoint array.
    1. Fill missing (zero) joints
    2. Smooth each joint-axis independently
    3. Resample to TARGET_LEN
    4. Normalise (centre + scale)
    Returns (TARGET_LEN, J, 3) float32.
    """
    T, J, C = raw.shape

    if T < MIN_FRAMES:
        raise ValueError(f"Sequence too short: {T} frames")

    # smooth
    for j in range(J):
        for c in range(C):
            raw[:, j, c] = _smooth(raw[:, j, c])

    # fill missing
    raw = _fill_missing(raw)

    # resample to TARGET_LEN
    resampled = np.zeros((TARGET_LEN, J, C), dtype=np.float32)
    for j in range(J):
        for c in range(C):
            resampled[:, j, c] = _interpolate_to_length(raw[:, j, c], TARGET_LEN)

    # normalise
    resampled = _normalise_keypoints(resampled)

    return resampled


# ─── UI-PRMD loader ───────────────────────────────────────────────────────────

def load_uiprmd_sample(filepath: str) -> np.ndarray:
    """
    Load one UI-PRMD Vicon positions file.
    Returns (T, 6, 3) array for the 6 lower-limb joints.
    File: T rows x 117 cols (39 joints x 3 axes, row-major: j0x j0y j0z j1x ...)
    Joint mapping (confirmed from data): LASI=23, LKNE=28, LANK=30, RASI=24, RKNE=34, RANK=36
    """
    df  = pd.read_csv(filepath, header=None)
    arr = df.values.astype(np.float32)   # (T, 66)

    T   = arr.shape[0]
    kps = np.zeros((T, N_JOINTS, 3), dtype=np.float32)

    for ji, jname in enumerate(JOINT_ORDER):
        col_base = JOINT_MAP[jname] * 3
        kps[:, ji, 0] = arr[:, col_base]       # x
        kps[:, ji, 1] = arr[:, col_base + 1]   # y
        kps[:, ji, 2] = arr[:, col_base + 2]   # z

    return kps   # (T, 6, 3)


def load_uiprmd_dataset(
    lower_limb_only: bool = True,
    use_kinect: bool = False,   # False = Vicon (confirmed correct joint mapping)
) -> List[dict]:
    """
    Load all UI-PRMD samples (correct + incorrect).
    Uses Vicon/Positions by default — confirmed joint mapping (LASI/LKNE/LANK).
    Kinect positions are local/relative coords and not suitable for ST-GCN.
    Returns list of sample dicts.
    """
    modality = 'Kinect' if use_kinect else 'Vicon'
    samples  = []

    for label_name, label_val, inc_suffix in [
        ('Segmented Movements',           1, ''),
        ('Incorrect Segmented Movements', 0, '_inc'),
    ]:
        pos_dir = os.path.join(BASE_DIR, label_name, modality, 'Positions')
        if not os.path.exists(pos_dir):
            print(f"WARNING: {pos_dir} not found — skipping")
            continue

        for fname in sorted(os.listdir(pos_dir)):
            if not fname.endswith('.txt'):
                continue

            # parse filename: m01_s01_e01_positions[_inc].txt
            parts    = fname.replace('_positions', '').replace(inc_suffix, '').replace('.txt', '').split('_')
            if len(parts) < 3:
                continue
            movement = parts[0]   # m01..m10
            subject  = parts[1]   # s01..s10
            exercise = parts[2]   # e01..e10

            if lower_limb_only and movement not in LOWER_LIMB_MOVEMENTS:
                continue

            fpath = os.path.join(pos_dir, fname)
            try:
                raw = load_uiprmd_sample(fpath)
                kps = process_keypoints(raw)
            except Exception as ex:
                print(f"SKIP {fname}: {ex}")
                continue

            samples.append({
                'keypoints': kps,             # (150, 6, 3)
                'adj':       ADJ,             # (6, 6)
                'label':     label_val,       # 1=correct, 0=incorrect
                'exercise':  MOVEMENT_MAP.get(movement, movement),
                'subject':   f'uiprmd_{subject}',
                'source':    'uiprmd',
                'movement':  movement,
            })

    print(f"UI-PRMD loaded: {len(samples)} samples")
    return samples


# ─── UCO PhyRehab loader ──────────────────────────────────────────────────────

UCO_JOINT_MAP = {
    'l_hip':   0,
    'l_knee':  1,
    'l_ankle': 2,
    'r_hip':   3,
    'r_knee':  4,
    'r_ankle': 5,
}

UCO_FRAME_KEYS = {
    'l_hip':   'l_hip',
    'l_knee':  'l_knee',
    'l_ankle': 'l_ankle',
    'r_hip':   'r_hip',
    'r_knee':  'r_knee',
    'r_ankle': 'r_ankle',
}


def load_uco_sample(sample: dict) -> np.ndarray:
    """
    Parse one UCO JSON sample into (T, 6, 3) keypoint array.
    Right-side joints are sparse (zero if missing — _fill_missing handles it).
    """
    frames = sample['frames']
    T      = len(frames)

    kps = np.zeros((T, N_JOINTS, 3), dtype=np.float32)

    for t, frame in enumerate(frames):
        joints = frame['joints']
        for ji, jname in enumerate(JOINT_ORDER):
            uco_key = UCO_FRAME_KEYS[jname]
            if uco_key in joints and isinstance(joints[uco_key], dict):
                jd = joints[uco_key]
                kps[t, ji, 0] = float(jd.get('x', 0.0))
                kps[t, ji, 1] = float(jd.get('y', 0.0))
                kps[t, ji, 2] = float(jd.get('z', 0.0))
            # else stays 0 — filled by _fill_missing

    return kps   # (T, 6, 3)


def load_uco_dataset() -> List[dict]:
    """
    Load all UCO PhyRehab samples.
    All UCO samples are correct reps (label=1).
    Quality score supervision added later via KIMORE fine-tuning.
    """
    with open(UCO_JSON) as f:
        data = json.load(f)['data']

    samples = []
    skipped = 0

    for i, sample in enumerate(data):
        if sample.get('body', '') != 'lower':
            continue   # skip upper-body samples

        try:
            raw = load_uco_sample(sample)
            kps = process_keypoints(raw)
        except Exception as ex:
            skipped += 1
            continue

        samples.append({
            'keypoints': kps,
            'adj':       ADJ,
            'label':     1,    # all correct reps in UCO
            'exercise':  f"uco_ex{sample.get('exercise', '00')}",
            'subject':   f"uco_folder{sample.get('folder', '0')}",
            'source':    'uco',
            'side':      sample.get('side', 'unknown'),
        })

    print(f"UCO loaded: {len(samples)} samples ({skipped} skipped)")
    return samples


# ─── master dataset builder ───────────────────────────────────────────────────

def build_master_dataset(
    lower_limb_only: bool = True,
    use_uco: bool = True,
) -> List[dict]:
    """
    Combine UI-PRMD + UCO into one list.
    This is the input to the PyTorch Dataset.
    """
    samples = load_uiprmd_dataset(lower_limb_only=lower_limb_only)
    if use_uco:
        samples += load_uco_dataset()

    print(f"\nMaster dataset: {len(samples)} total samples")
    print(f"  Correct:   {sum(s['label']==1 for s in samples)}")
    print(f"  Incorrect: {sum(s['label']==0 for s in samples)}")
    print(f"  Sources:   { {s['source'] for s in samples} }")
    return samples


# ─── PyTorch Dataset ─────────────────────────────────────────────────────────

class RehabDataset(Dataset):
    """
    PyTorch Dataset wrapping the master sample list.
    Returns tensors ready for ST-GCN → Transformer.
    """

    def __init__(self, samples: List[dict], augment: bool = False):
        self.samples = samples
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s   = self.samples[idx]
        kps = s['keypoints'].copy()   # (150, 6, 3)

        if self.augment:
            kps = self._augment(kps)

        # ST-GCN expects (C, T, V) = (3, 150, 6)
        kps_tensor = torch.from_numpy(kps).permute(2, 0, 1).float()   # (3, 150, 6)
        adj_tensor = torch.from_numpy(s['adj']).float()                # (6, 6)
        label      = torch.tensor(s['label'], dtype=torch.long)

        return {
            'keypoints': kps_tensor,   # (3, 150, 6)
            'adj':       adj_tensor,   # (6, 6)
            'label':     label,
            'exercise':  s['exercise'],
            'subject':   s['subject'],
            'source':    s['source'],
        }

    def _augment(self, kps: np.ndarray) -> np.ndarray:
        """
        On-the-fly augmentation.
        kps: (150, 6, 3)
        """
        # 1. temporal jitter ±20%
        if np.random.rand() < 0.5:
            factor  = np.random.uniform(0.8, 1.2)
            new_len = int(TARGET_LEN * factor)
            new_len = max(new_len, MIN_FRAMES)
            stretched = np.zeros((new_len, N_JOINTS, 3), dtype=np.float32)
            for j in range(N_JOINTS):
                for c in range(3):
                    stretched[:, j, c] = _interpolate_to_length(kps[:, j, c], new_len)
            kps = np.zeros((TARGET_LEN, N_JOINTS, 3), dtype=np.float32)
            for j in range(N_JOINTS):
                for c in range(3):
                    kps[:, j, c] = _interpolate_to_length(stretched[:, j, c], TARGET_LEN)

        # 2. mirror flip (swap L↔R)
        if np.random.rand() < 0.5:
            # swap: L_hip(0)↔R_hip(3), L_knee(1)↔R_knee(4), L_ankle(2)↔R_ankle(5)
            kps[:, [0,1,2,3,4,5], :] = kps[:, [3,4,5,0,1,2], :]
            kps[:, :, 0] *= -1.0   # flip x axis

        # 3. Gaussian noise σ=2mm (positions already normalised — use 0.02)
        if np.random.rand() < 0.5:
            kps += np.random.randn(*kps.shape).astype(np.float32) * 0.02

        # 4. small rotation around vertical axis (±10°)
        if np.random.rand() < 0.3:
            theta = np.radians(np.random.uniform(-10, 10))
            cos_t, sin_t = np.cos(theta), np.sin(theta)
            x = kps[:, :, 0].copy()
            z = kps[:, :, 2].copy()
            kps[:, :, 0] = cos_t * x - sin_t * z
            kps[:, :, 2] = sin_t * x + cos_t * z

        return kps


# ─── LOSO split ───────────────────────────────────────────────────────────────

def loso_split(
    samples: List[dict],
    held_out_subject: str
) -> Tuple[List[dict], List[dict]]:
    """
    Leave-One-Subject-Out split.
    Use 'uiprmd_s01' .. 'uiprmd_s10' as held_out_subject.
    UCO samples always go to train (no subject overlap with UI-PRMD).
    """
    train = [s for s in samples if s['subject'] != held_out_subject]
    test  = [s for s in samples if s['subject'] == held_out_subject]
    print(f"LOSO held-out: {held_out_subject} → train={len(train)}, test={len(test)}")
    return train, test


def get_all_uiprmd_subjects(samples: List[dict]) -> List[str]:
    return sorted({s['subject'] for s in samples if s['source'] == 'uiprmd'})


# ─── scalar feature extractor (XGBoost input) ────────────────────────────────

def extract_scalar_features(kps: np.ndarray) -> np.ndarray:
    """
    Compute 10 scalar features from (TARGET_LEN, 6, 3) keypoints.
    These go directly to XGBoost, bypassing ST-GCN.

    Features:
      0  L knee ROM (degrees, computed from positions)
      1  R knee ROM
      2  L knee peak angle
      3  R knee peak angle
      4  L/R ROM symmetry index (%)
      5  max angular velocity (°/s)
      6  mean jerk proxy (position jerk norm)
      7  trunk lean proxy (hip-spine deviation at peak)
      8  L/R temporal lag (frames)
      9  smoothness proxy (1 / position jerk std)
    """
    def _angle_from_positions(a, b, c):
        """Angle at joint b given positions a, b, c. Returns degrees array."""
        ba = a - b
        bc = c - b
        cos_ang = np.sum(ba * bc, axis=1) / (
            np.linalg.norm(ba, axis=1) * np.linalg.norm(bc, axis=1) + 1e-6
        )
        return np.degrees(np.arccos(np.clip(cos_ang, -1, 1)))

    # joint indices: L_hip=0, L_knee=1, L_ankle=2, R_hip=3, R_knee=4, R_ankle=5
    l_knee_ang = _angle_from_positions(kps[:, 0, :], kps[:, 1, :], kps[:, 2, :])
    r_knee_ang = _angle_from_positions(kps[:, 3, :], kps[:, 4, :], kps[:, 5, :])

    l_rom = float(np.max(l_knee_ang) - np.min(l_knee_ang))
    r_rom = float(np.max(r_knee_ang) - np.min(r_knee_ang))
    l_peak = float(np.min(l_knee_ang))
    r_peak = float(np.min(r_knee_ang))
    sym   = float(abs(l_rom - r_rom) / ((l_rom + r_rom) / 2.0 + 1e-6) * 100.0)

    # angular velocity proxy (mean frame-to-frame angle change)
    l_vel    = np.abs(np.diff(l_knee_ang)) * 30.0   # FPS=30
    max_vel  = float(np.max(l_vel))

    # jerk proxy (norm of 2nd derivative of hip position)
    hip_mid  = (kps[:, 0, :] + kps[:, 3, :]) / 2.0
    jerk_vec = np.diff(np.diff(hip_mid, axis=0), axis=0)
    jerk_norm= np.linalg.norm(jerk_vec, axis=1)
    mean_jerk= float(np.mean(jerk_norm))

    # trunk lean proxy: hip midpoint x-deviation at peak knee bend
    peak_idx    = int(np.argmin(l_knee_ang))
    trunk_lean  = float(abs(hip_mid[peak_idx, 0] - hip_mid[0, 0]))

    # temporal lag between L and R knee angle curves
    corr  = np.correlate(l_knee_ang - l_knee_ang.mean(),
                         r_knee_ang - r_knee_ang.mean(), mode='full')
    lag   = float(abs(np.argmax(corr) - (len(l_knee_ang) - 1)))

    smoothness = float(1.0 / (np.std(jerk_norm) + 1e-6))

    return np.array([
        l_rom / 180.0,
        r_rom / 180.0,
        l_peak / 180.0,
        r_peak / 180.0,
        sym / 100.0,
        max_vel / 200.0,
        mean_jerk / 10.0,
        trunk_lean,
        lag / TARGET_LEN,
        np.clip(smoothness / 100.0, 0, 1),
    ], dtype=np.float32)


# ─── ST-GCN block ─────────────────────────────────────────────────────────────

class STGCNBlock(nn.Module):
    """
    One ST-GCN block: spatial GCN + temporal conv.
    Input : (B, C_in, T, V)
    Output: (B, C_out, T, V)
    """

    def __init__(self, c_in: int, c_out: int, kernel_size: int = 9, stride: int = 1):
        super().__init__()
        pad = (kernel_size - 1) // 2

        self.gcn = nn.Linear(c_in, c_out)   # spatial (applied per node)
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(c_out),
            nn.ReLU(),
            nn.Conv2d(c_out, c_out, kernel_size=(kernel_size, 1),
                      stride=(stride, 1), padding=(pad, 0)),
            nn.BatchNorm2d(c_out),
            nn.Dropout(0.1),
        )
        self.residual = (
            nn.Sequential(
                nn.Conv2d(c_in, c_out, kernel_size=1, stride=(stride, 1)),
                nn.BatchNorm2d(c_out),
            ) if c_in != c_out or stride != 1 else nn.Identity()
        )
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
        # x: (B, C, T, V),  A: (V, V) or (B, V, V)
        B, C, T, V = x.shape

        # ensure A is always (V, V) — take first element if batched
        if A.dim() == 3:
            A = A[0]   # (V, V)

        # spatial GCN: aggregate neighbour features via A
        # reshape to (B*T, V, C), multiply by A, apply linear
        x_s = x.permute(0, 2, 3, 1).contiguous().view(B * T, V, C)   # (B*T, V, C)
        x_s = torch.bmm(A.unsqueeze(0).expand(B * T, -1, -1), x_s)   # (B*T, V, C)
        x_s = self.gcn(x_s)                                            # (B*T, V, C_out)
        x_s = x_s.view(B, T, V, -1).permute(0, 3, 1, 2)               # (B, C_out, T, V)

        # temporal conv
        out = self.tcn(x_s) + self.residual(x)
        return self.relu(out)


class STGCN(nn.Module):
    """
    3-block ST-GCN spatial encoder.
    Input : (B, 3, T, 6)    — (batch, xyz, frames, joints)
    Output: (B, T, 128)     — frame-level features for Transformer
    """

    def __init__(self):
        super().__init__()
        self.blocks = nn.ModuleList([
            STGCNBlock(3,   32,  kernel_size=9),
            STGCNBlock(32,  64,  kernel_size=9),
            STGCNBlock(64, 128,  kernel_size=9),
        ])
        self.bn_input = nn.BatchNorm1d(3 * 6)   # normalise input channels×joints

    def forward(self, x: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
        # x: (B, 3, T, 6),  A: (6, 6)
        B, C, T, V = x.shape

        # input BN
        x_flat = x.permute(0, 2, 1, 3).contiguous().view(B * T, C * V)
        x_flat = self.bn_input(x_flat)
        x = x_flat.view(B, T, C, V).permute(0, 2, 1, 3)   # back to (B, C, T, V)

        for block in self.blocks:
            x = block(x, A)   # (B, 128, T, 6)

        # pool over joints → frame-level features
        x = x.mean(dim=3)          # (B, 128, T)
        x = x.permute(0, 2, 1)    # (B, T, 128)
        return x


# ─── Two-Layer Clinical Transformer ──────────────────────────────────────────

class ClinicalTransformer(nn.Module):
    """
    Two-layer transformer with explicit clinical interpretation per layer.

    Layer 1 — Temporal correlation:
      Self-attention across T=150 frames.
      Learns which frames are related (e.g. frame 45 ↔ frame 120 = both peaks).
      Input: frame-level ST-GCN features (B, T, 128).

    Layer 2 — Clinical interpretation:
      Self-attention over the temporally-aware representations.
      Learns what the temporal correlations mean clinically
      (e.g. 'peak valgus coincides with peak velocity = dangerous pattern').
      CLS token aggregates the full rep into a single vector.

    Output: (B, 128) rep-level summary vector → XGBoost / quality head.
    """

    def __init__(
        self,
        d_model: int = 128,
        n_heads: int = 4,
        d_ff: int = 256,
        dropout: float = 0.1,
    ):
        super().__init__()

        # positional encoding
        self.pos_enc = self._build_pos_enc(TARGET_LEN + 1, d_model)  # +1 for CLS

        # CLS token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

        # Layer 1: temporal correlation
        self.layer1 = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True,
            norm_first=True,   # pre-norm for stability
        )
        self.norm1 = nn.LayerNorm(d_model)

        # Layer 2: clinical interpretation
        self.layer2 = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True,
            norm_first=True,
        )
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def _build_pos_enc(max_len: int, d_model: int) -> nn.Parameter:
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() *
                        (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        return nn.Parameter(pe.unsqueeze(0), requires_grad=False)  # (1, max_len, d_model)

    def forward(
        self,
        x: torch.Tensor,
        return_attention: bool = False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        x: (B, T, 128) ST-GCN frame features
        Returns: (B, 128) CLS vector, optionally layer-2 attention weights
        """
        B, T, D = x.shape

        # prepend CLS token
        cls  = self.cls_token.expand(B, -1, -1)        # (B, 1, 128)
        x    = torch.cat([cls, x], dim=1)               # (B, T+1, 128)
        x    = x + self.pos_enc[:, :T + 1, :]
        x    = self.dropout(x)

        # Layer 1: temporal correlation
        x    = self.layer1(x)
        x    = self.norm1(x)

        # Layer 2: clinical interpretation
        # hook attention weights for therapist dashboard heatmap
        attn_weights = None
        if return_attention:
            # manually run attention to extract weights
            x2, attn_weights = self._layer2_with_attn(x)
            x = self.norm2(x2)
        else:
            x = self.layer2(x)
            x = self.norm2(x)

        # CLS token = rep summary
        cls_out = x[:, 0, :]   # (B, 128)
        return cls_out, attn_weights

    def _layer2_with_attn(
        self, x: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Run layer2 and return (output, attention_weights)."""
        # access layer2 self-attention directly
        sa  = self.layer2.self_attn
        x_n = self.layer2.norm1(x) if self.layer2.norm_first else x
        attn_out, attn_w = sa(x_n, x_n, x_n, need_weights=True, average_attn_weights=True)
        # continue through the rest of the encoder layer manually
        x2  = x + self.layer2.dropout1(attn_out)
        x_n2 = self.layer2.norm2(x2) if self.layer2.norm_first else x2
        ff_out = self.layer2.linear2(
            self.layer2.dropout(self.layer2.activation(self.layer2.linear1(x_n2)))
        )
        x2 = x2 + self.layer2.dropout2(ff_out)
        return x2, attn_w   # attn_w: (B, T+1, T+1)


# ─── full model ───────────────────────────────────────────────────────────────

class RehabNet(nn.Module):
    """
    Full pipeline: ST-GCN → Two-Layer Clinical Transformer → Classification head.
    For XGBoost head: call encode() to get (B, 128) embeddings.
    """

    def __init__(self, n_classes: int = 2):
        super().__init__()
        self.stgcn       = STGCN()
        self.transformer = ClinicalTransformer()
        self.classifier  = nn.Sequential(
            nn.Linear(128 + 10, 64),   # 128 CLS + 10 scalar features
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, n_classes),
        )

    def encode(
        self,
        keypoints: torch.Tensor,
        adj: torch.Tensor,
        return_attention: bool = False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        Returns (B, 128) rep embedding + optional attention weights.
        adj can be (6,6) or (B,6,6) — reduced to (6,6) inside STGCNBlock.
        """
        if adj.dim() == 3:
            adj = adj[0]   # (6, 6)
        frame_feats = self.stgcn(keypoints, adj)
        cls_vec, attn = self.transformer(frame_feats, return_attention)
        return cls_vec, attn

    def forward(
        self,
        keypoints: torch.Tensor,
        adj: torch.Tensor,
        scalars: torch.Tensor,
    ) -> torch.Tensor:
        """
        keypoints : (B, 3, T, 6)
        adj       : (B, 6, 6) or (6, 6)
        scalars   : (B, 10)
        Returns   : (B, n_classes) logits
        """
        cls_vec, _ = self.encode(keypoints, adj)
        combined   = torch.cat([cls_vec, scalars], dim=1)
        return self.classifier(combined)


# ─── DataLoader builder ───────────────────────────────────────────────────────

def build_dataloaders(
    train_samples: List[dict],
    test_samples:  List[dict],
    batch_size: int = 32,
) -> Tuple[DataLoader, DataLoader]:
    """Build train/test DataLoaders with scalar features attached."""

    # attach scalar features to each sample
    for s in train_samples + test_samples:
        s['scalars'] = extract_scalar_features(s['keypoints'])

    train_ds = RehabDataset(train_samples, augment=True)
    test_ds  = RehabDataset(test_samples,  augment=False)

    # custom collate to handle scalar features
    def collate(batch):
        return {
            'keypoints': torch.stack([b['keypoints'] for b in batch]),
            'adj':       torch.stack([b['adj']       for b in batch]),
            'label':     torch.stack([b['label']     for b in batch]),
            'scalars':   torch.tensor(
                             np.stack([b['scalars'] for b in batch]),
                             dtype=torch.float32
                         ) if 'scalars' in batch[0] else None,
            'exercise':  [b['exercise'] for b in batch],
            'subject':   [b['subject']  for b in batch],
        }

    train_dl = DataLoader(train_ds, batch_size=batch_size,
                          shuffle=True,  collate_fn=collate, num_workers=2)
    test_dl  = DataLoader(test_ds,  batch_size=batch_size,
                          shuffle=False, collate_fn=collate, num_workers=2)
    return train_dl, test_dl


# ─── training loop ────────────────────────────────────────────────────────────

def train_one_epoch(
    model: RehabNet,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> float:
    model.train()
    total_loss = 0.0
    adj_shared = torch.from_numpy(ADJ).to(device)

    for batch in loader:
        kps     = batch['keypoints'].to(device)   # (B, 3, 150, 6)
        labels  = batch['label'].to(device)
        scalars = batch['scalars'].to(device) if batch['scalars'] is not None \
                  else torch.zeros(kps.shape[0], 10, device=device)

        logits = model(kps, adj_shared, scalars)
        loss   = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(
    model: RehabNet,
    loader: DataLoader,
    device: torch.device,
) -> dict:
    model.eval()
    adj_shared = torch.from_numpy(ADJ).to(device)
    all_preds, all_labels = [], []

    for batch in loader:
        kps     = batch['keypoints'].to(device)
        labels  = batch['label'].to(device)
        scalars = batch['scalars'].to(device) if batch['scalars'] is not None \
                  else torch.zeros(kps.shape[0], 10, device=device)

        logits = model(kps, adj_shared, scalars)
        preds  = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    from sklearn.metrics import accuracy_score, f1_score, classification_report
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro')
    print(classification_report(all_labels, all_preds,
                                 target_names=['incorrect', 'correct']))
    return {'accuracy': acc, 'f1_macro': f1}


# ─── LOSO training runner ─────────────────────────────────────────────────────

def run_loso_training(
    n_epochs: int = 30,
    batch_size: int = 32,
    lr: float = 1e-3,
    device_str: str = 'cuda' if torch.cuda.is_available() else 'cpu',
):
    """
    Full LOSO cross-validation run.
    Trains one model per UI-PRMD subject, reports mean ± std accuracy.
    """
    device  = torch.device(device_str)
    samples = build_master_dataset(lower_limb_only=True, use_uco=True)
    subjects = get_all_uiprmd_subjects(samples)

    results = []
    for subj in subjects:
        print(f"\n{'='*50}")
        print(f"LOSO fold: held-out = {subj}")
        train_s, test_s = loso_split(samples, subj)
        train_dl, test_dl = build_dataloaders(train_s, test_s, batch_size)

        model     = RehabNet(n_classes=2).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

        for epoch in range(n_epochs):
            loss = train_one_epoch(model, train_dl, optimizer, device)
            scheduler.step()
            if (epoch + 1) % 10 == 0:
                print(f"  Epoch {epoch+1}/{n_epochs}  loss={loss:.4f}")

        metrics = evaluate(model, test_dl, device)
        results.append(metrics)
        print(f"  → acc={metrics['accuracy']:.3f}  f1={metrics['f1_macro']:.3f}")

    accs = [r['accuracy'] for r in results]
    print(f"\nLOSO summary: acc = {np.mean(accs):.3f} ± {np.std(accs):.3f}")
    return results


# ─── smoke test ───────────────────────────────────────────────────────────────

if __name__ == '__main__':
    print("── Architecture smoke test (no data needed) ────────────────")
    device = torch.device('cpu')
    B = 4

    kps     = torch.randn(B, 3, TARGET_LEN, N_JOINTS)
    adj     = torch.from_numpy(ADJ).unsqueeze(0).expand(B, -1, -1)
    scalars = torch.randn(B, 10)

    model  = RehabNet(n_classes=2)
    logits = model(kps, adj, scalars)
    print(f"  Input  kps:     {kps.shape}")
    print(f"  Input  adj:     {adj.shape}")
    print(f"  Input  scalars: {scalars.shape}")
    print(f"  Output logits:  {logits.shape}")

    # test attention extraction
    cls_vec, attn = model.encode(kps, adj, return_attention=True)
    print(f"  CLS embedding:  {cls_vec.shape}")
    print(f"  Attn weights:   {attn.shape if attn is not None else None}")

    # adjacency matrix
    print(f"\n── Adjacency matrix (normalised) ───────────────────────────")
    print(np.round(ADJ, 3))

    print("\n── Smoke test passed ───────────────────────────────────────")
    print("\nTo load data and run LOSO, call:")
    print("  results = run_loso_training(n_epochs=30, batch_size=32)")

── Architecture smoke test (no data needed) ────────────────
  Input  kps:     torch.Size([4, 3, 150, 6])
  Input  adj:     torch.Size([4, 6, 6])
  Input  scalars: torch.Size([4, 10])
  Output logits:  torch.Size([4, 2])
  CLS embedding:  torch.Size([4, 128])
  Attn weights:   torch.Size([4, 151, 151])

── Adjacency matrix (normalised) ───────────────────────────
[[0.333 0.289 0.    0.333 0.    0.   ]
 [0.289 0.25  0.289 0.    0.25  0.   ]
 [0.    0.289 0.333 0.    0.    0.333]
 [0.333 0.    0.    0.333 0.289 0.   ]
 [0.    0.25  0.    0.289 0.25  0.289]
 [0.    0.    0.333 0.    0.289 0.333]]

── Smoke test passed ───────────────────────────────────────

To load data and run LOSO, call:
  results = run_loso_training(n_epochs=30, batch_size=32)


In [29]:
samples = build_master_dataset(lower_limb_only=True, use_uco=True)

s = samples[0]
kps = s['keypoints']
print("l_hip  z:", kps[:3, 0, 2])    # should be positive, ~0.1-0.5
print("l_knee z:", kps[:3, 1, 2])    # should be < l_hip z
print("l_ankle z:", kps[:3, 2, 2])   # should be lowest, near 0 or negative
print("variance across frames:", kps[:, 0, :].var(axis=0))  # should be non-zero

UI-PRMD loaded: 1180 samples
UCO loaded: 212 samples (4 skipped)

Master dataset: 1392 total samples
  Correct:   802
  Incorrect: 590
  Sources:   {'uiprmd', 'uco'}
l_hip  z: [0.37325454 0.3676207  0.3610767 ]
l_knee z: [-0.25935283 -0.2587353  -0.258473  ]
l_ankle z: [-0.9452087  -0.94536364 -0.94551516]
variance across frames: [8.6213648e-03 9.7202930e-05 8.9676484e-02]


In [2]:
"""
=============================================================================
PS2 — Data Loading + ST-GCN + Two-Layer Clinical Transformer Pipeline
=============================================================================
Datasets:
  UI-PRMD  : Kinect Positions (66 cols = 22 joints × 3 xyz)
             correct   → Segmented Movements/Kinect/Positions/
             incorrect → Incorrect Segmented Movements/Kinect/Positions/
             filename  : m{mov:02d}_s{subj:02d}_e{ex:02d}_positions[_inc].txt
             label     : 1 = correct, 0 = incorrect

  UCO      : dataset_3d_with_angles.json
             432 samples, per-frame l_hip/l_knee/l_ankle (+ r_* sparse)
             label     : all are correct reps (quality supervision later)

Output per sample (transformer-ready):
  keypoints : (TARGET_LEN, N_JOINTS, 3)  float32   ← ST-GCN input
  adj       : (N_JOINTS, N_JOINTS)        float32   ← fixed anatomical graph
  label     : int                                   ← 0=incorrect, 1=correct
  exercise  : str
  subject   : str                                   ← for LOSO split
  source    : str                                   ← 'uiprmd' | 'uco'

UI-PRMD Kinect joint index map (22 joints, 0-indexed):
  0  SpineBase      1  SpineMid       2  Neck           3  Head
  4  ShoulderLeft   5  ElbowLeft      6  WristLeft      7  HandLeft
  8  ShoulderRight  9  ElbowRight     10 WristRight     11 HandRight
  12 HipLeft        13 KneeLeft       14 AnkleLeft      15 FootLeft
  16 HipRight       17 KneeRight      18 AnkleRight     19 FootRight
  20 SpineShoulder  21 HandTipLeft    (some versions vary — validated below)

Lower-limb joints used (6):
  LEFT  : HipLeft(12), KneeLeft(13),  AnkleLeft(14)
  RIGHT : HipRight(16), KneeRight(17), AnkleRight(18)
"""

import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.signal import savgol_filter
from scipy.interpolate import interp1d
from typing import Dict, List, Optional, Tuple


# ─── config ──────────────────────────────────────────────────────────────────

BASE_DIR      = '/content/drive/MyDrive/datasets'
UCO_JSON      = os.path.join(BASE_DIR, 'dataset_3d_with_angles.json')
TARGET_LEN    = 150       # fixed sequence length for all samples
N_JOINTS      = 6         # lower-limb only: L_hip, L_knee, L_ankle, R_hip, R_knee, R_ankle
IN_CHANNELS   = 3         # x, y, z
SG_WIN        = 7
SG_POLY       = 2
MIN_FRAMES    = 20

# UI-PRMD: Vicon Positions (39 joints x 3 = 117 cols)
# Confirmed from data inspection (z = vertical axis):
#   LASI(23) z=693  LKNE(28) z=474  LANK(30) z=87   ← left chain
#   RASI(24) z=686  RKNE(34) z=465  RANK(36) z=91   ← right chain
# Using LASI/RASI as hip proxies (anterior superior iliac spine)
JOINT_MAP = {
    'l_hip'  : 23,   # LASI — left anterior superior iliac spine
    'l_knee' : 28,   # LKNE — left knee
    'l_ankle': 30,   # LANK — left ankle
    'r_hip'  : 24,   # RASI — right anterior superior iliac spine
    'r_knee' : 34,   # RKNE — right knee
    'r_ankle': 36,   # RANK — right ankle
}
JOINT_ORDER = ['l_hip', 'l_knee', 'l_ankle', 'r_hip', 'r_knee', 'r_ankle']

# UI-PRMD movement → exercise name mapping
MOVEMENT_MAP = {
    'm01': 'deep_squat',
    'm02': 'hurdle_step',
    'm03': 'inline_lunge',
    'm04': 'side_lunge',       # lower limb relevant
    'm05': 'sit_to_stand',
    'm06': 'standing_active_straight_leg_raise',
    'm07': 'standing_shoulder_abduction',   # upper — include but low weight
    'm08': 'standing_shoulder_extension',   # upper
    'm09': 'standing_shoulder_internal_rotation',  # upper
    'm10': 'standing_shoulder_scaption',    # upper
}

# Lower-limb relevant movements (use for lower-limb model training)
LOWER_LIMB_MOVEMENTS = {'m01', 'm02', 'm03', 'm04', 'm05', 'm06'}


# ─── anatomical graph (ST-GCN adjacency) ─────────────────────────────────────

def build_adjacency() -> np.ndarray:
    """
    Fixed anatomical adjacency matrix for 6 lower-limb joints.
    Joint order: [L_hip(0), L_knee(1), L_ankle(2), R_hip(3), R_knee(4), R_ankle(5)]

    Edges:
      Anatomical bones  : L_hip-L_knee, L_knee-L_ankle (left chain)
                          R_hip-R_knee, R_knee-R_ankle (right chain)
      Bilateral links   : L_hip-R_hip  (pelvis)
                          L_knee-R_knee, L_ankle-R_ankle (cross-body)
      Self-connections  : all diagonal = 1 (standard ST-GCN)
    """
    A = np.zeros((N_JOINTS, N_JOINTS), dtype=np.float32)

    edges = [
        (0, 1),   # L_hip   → L_knee
        (1, 2),   # L_knee  → L_ankle
        (3, 4),   # R_hip   → R_knee
        (4, 5),   # R_knee  → R_ankle
        (0, 3),   # L_hip   ↔ R_hip   (pelvis)
        (1, 4),   # L_knee  ↔ R_knee  (cross)
        (2, 5),   # L_ankle ↔ R_ankle (cross)
    ]

    for i, j in edges:
        A[i, j] = 1.0
        A[j, i] = 1.0

    # self-connections
    np.fill_diagonal(A, 1.0)

    # symmetric normalisation: D^{-1/2} A D^{-1/2}
    D = np.diag(A.sum(axis=1))
    D_inv_sqrt = np.diag(1.0 / np.sqrt(A.sum(axis=1) + 1e-6))
    A_norm = D_inv_sqrt @ A @ D_inv_sqrt

    return A_norm.astype(np.float32)


ADJ = build_adjacency()   # (6, 6) — shared across all samples


# ─── signal processing helpers ───────────────────────────────────────────────

def _smooth(arr: np.ndarray) -> np.ndarray:
    if len(arr) < SG_WIN + 2:
        return arr
    win = SG_WIN if len(arr) > SG_WIN else (len(arr) if len(arr) % 2 == 1 else len(arr) - 1)
    win = max(win, SG_POLY + 2 if (SG_POLY + 2) % 2 == 1 else SG_POLY + 3)
    return savgol_filter(arr, window_length=win, polyorder=SG_POLY)


def _interpolate_to_length(arr: np.ndarray, target: int) -> np.ndarray:
    """Resample 1-D array to target length."""
    if len(arr) == target:
        return arr
    x_old = np.linspace(0, 1, len(arr))
    x_new = np.linspace(0, 1, target)
    return np.interp(x_new, x_old, arr)


def _normalise_keypoints(kps: np.ndarray) -> np.ndarray:
    """
    Normalise skeleton keypoints:
      1. Centre on mean hip midpoint (single reference point across all frames)
         — removes subject position in room, keeps inter-frame motion intact
      2. Scale by mean hip-to-ankle distance (full leg length)
         — removes subject height differences, preserves joint ratios

    kps: (T, J, 3)  joints: L_hip=0, L_knee=1, L_ankle=2, R_hip=3, R_knee=4, R_ankle=5
    Returns normalised (T, J, 3) where:
      - hip midpoint at origin on average
      - full leg length ~1.0
      - joints still move freely across frames
    """
    # use MEAN hip midpoint as single anchor (not per-frame — preserves motion)
    hip_mid_mean = ((kps[:, 0, :] + kps[:, 3, :]) / 2.0).mean(axis=0)  # (3,)
    kps = kps - hip_mid_mean[np.newaxis, np.newaxis, :]                  # (T, J, 3)

    # scale by mean hip-to-ankle distance (full leg length — more stable than hip-knee)
    l_leg = np.mean(np.linalg.norm(kps[:, 0, :] - kps[:, 2, :], axis=1))
    r_leg = np.mean(np.linalg.norm(kps[:, 3, :] - kps[:, 5, :], axis=1))
    leg_len = (l_leg + r_leg) / 2.0 + 1e-6
    kps = kps / leg_len

    return kps.astype(np.float32)


def _fill_missing(kps: np.ndarray) -> np.ndarray:
    """
    Linear interpolation for zero-columns (missing joints in UCO right side).
    kps: (T, J, 3)
    """
    T, J, C = kps.shape
    for j in range(J):
        for c in range(C):
            col = kps[:, j, c]
            zero_mask = (col == 0.0)
            if zero_mask.all():
                # entire joint missing — mirror from opposite side
                opp = j + 3 if j < 3 else j - 3
                kps[:, j, c] = kps[:, opp, c]
            elif zero_mask.any():
                # partial — interpolate
                valid = ~zero_mask
                if valid.sum() >= 2:
                    f = interp1d(np.where(valid)[0], col[valid],
                                 bounds_error=False, fill_value='extrapolate')
                    kps[:, j, c] = f(np.arange(T))
    return kps


def process_keypoints(raw: np.ndarray) -> np.ndarray:
    """
    Full preprocessing pipeline for (T, J, 3) keypoint array.
    1. Fill missing (zero) joints
    2. Smooth each joint-axis independently
    3. Resample to TARGET_LEN
    4. Normalise (centre + scale)
    Returns (TARGET_LEN, J, 3) float32.
    """
    T, J, C = raw.shape

    if T < MIN_FRAMES:
        raise ValueError(f"Sequence too short: {T} frames")

    # smooth
    for j in range(J):
        for c in range(C):
            raw[:, j, c] = _smooth(raw[:, j, c])

    # fill missing
    raw = _fill_missing(raw)

    # resample to TARGET_LEN
    resampled = np.zeros((TARGET_LEN, J, C), dtype=np.float32)
    for j in range(J):
        for c in range(C):
            resampled[:, j, c] = _interpolate_to_length(raw[:, j, c], TARGET_LEN)

    # normalise
    resampled = _normalise_keypoints(resampled)

    return resampled


# ─── UI-PRMD loader ───────────────────────────────────────────────────────────

def load_uiprmd_sample(filepath: str) -> np.ndarray:
    """
    Load one UI-PRMD Vicon positions file.
    Returns (T, 6, 3) array for the 6 lower-limb joints.
    File: T rows x 117 cols (39 joints x 3 axes, row-major: j0x j0y j0z j1x ...)
    Joint mapping (confirmed from data): LASI=23, LKNE=28, LANK=30, RASI=24, RKNE=34, RANK=36
    """
    df  = pd.read_csv(filepath, header=None)
    arr = df.values.astype(np.float32)   # (T, 66)

    T   = arr.shape[0]
    kps = np.zeros((T, N_JOINTS, 3), dtype=np.float32)

    for ji, jname in enumerate(JOINT_ORDER):
        col_base = JOINT_MAP[jname] * 3
        kps[:, ji, 0] = arr[:, col_base]       # x
        kps[:, ji, 1] = arr[:, col_base + 1]   # y
        kps[:, ji, 2] = arr[:, col_base + 2]   # z

    return kps   # (T, 6, 3)


def load_uiprmd_dataset(
    lower_limb_only: bool = True,
    use_kinect: bool = False,   # False = Vicon (confirmed correct joint mapping)
) -> List[dict]:
    """
    Load all UI-PRMD samples (correct + incorrect).
    Uses Vicon/Positions by default — confirmed joint mapping (LASI/LKNE/LANK).
    Kinect positions are local/relative coords and not suitable for ST-GCN.
    Returns list of sample dicts.
    """
    modality = 'Kinect' if use_kinect else 'Vicon'
    samples  = []

    for label_name, label_val, inc_suffix in [
        ('Segmented Movements',           1, ''),
        ('Incorrect Segmented Movements', 0, '_inc'),
    ]:
        pos_dir = os.path.join(BASE_DIR, label_name, modality, 'Positions')
        if not os.path.exists(pos_dir):
            print(f"WARNING: {pos_dir} not found — skipping")
            continue

        for fname in sorted(os.listdir(pos_dir)):
            if not fname.endswith('.txt'):
                continue

            # parse filename: m01_s01_e01_positions[_inc].txt
            parts    = fname.replace('_positions', '').replace(inc_suffix, '').replace('.txt', '').split('_')
            if len(parts) < 3:
                continue
            movement = parts[0]   # m01..m10
            subject  = parts[1]   # s01..s10
            exercise = parts[2]   # e01..e10

            if lower_limb_only and movement not in LOWER_LIMB_MOVEMENTS:
                continue

            fpath = os.path.join(pos_dir, fname)
            try:
                raw = load_uiprmd_sample(fpath)
                kps = process_keypoints(raw)
            except Exception as ex:
                print(f"SKIP {fname}: {ex}")
                continue

            samples.append({
                'keypoints': kps,             # (150, 6, 3)
                'adj':       ADJ,             # (6, 6)
                'label':     label_val,       # 1=correct, 0=incorrect
                'exercise':  MOVEMENT_MAP.get(movement, movement),
                'subject':   f'uiprmd_{subject}',
                'source':    'uiprmd',
                'movement':  movement,
            })

    print(f"UI-PRMD loaded: {len(samples)} samples")
    return samples


# ─── UCO PhyRehab loader ──────────────────────────────────────────────────────

UCO_JOINT_MAP = {
    'l_hip':   0,
    'l_knee':  1,
    'l_ankle': 2,
    'r_hip':   3,
    'r_knee':  4,
    'r_ankle': 5,
}

UCO_FRAME_KEYS = {
    'l_hip':   'l_hip',
    'l_knee':  'l_knee',
    'l_ankle': 'l_ankle',
    'r_hip':   'r_hip',
    'r_knee':  'r_knee',
    'r_ankle': 'r_ankle',
}


def load_uco_sample(sample: dict) -> np.ndarray:
    """
    Parse one UCO JSON sample into (T, 6, 3) keypoint array.
    Right-side joints are sparse (zero if missing — _fill_missing handles it).
    """
    frames = sample['frames']
    T      = len(frames)

    kps = np.zeros((T, N_JOINTS, 3), dtype=np.float32)

    for t, frame in enumerate(frames):
        joints = frame['joints']
        for ji, jname in enumerate(JOINT_ORDER):
            uco_key = UCO_FRAME_KEYS[jname]
            if uco_key in joints and isinstance(joints[uco_key], dict):
                jd = joints[uco_key]
                kps[t, ji, 0] = float(jd.get('x', 0.0))
                kps[t, ji, 1] = float(jd.get('y', 0.0))
                kps[t, ji, 2] = float(jd.get('z', 0.0))
            # else stays 0 — filled by _fill_missing

    return kps   # (T, 6, 3)


def load_uco_dataset() -> List[dict]:
    """
    Load all UCO PhyRehab samples.
    All UCO samples are correct reps (label=1).
    Quality score supervision added later via KIMORE fine-tuning.
    """
    with open(UCO_JSON) as f:
        data = json.load(f)['data']

    samples = []
    skipped = 0

    for i, sample in enumerate(data):
        if sample.get('body', '') != 'lower':
            continue   # skip upper-body samples

        try:
            raw = load_uco_sample(sample)
            kps = process_keypoints(raw)
        except Exception as ex:
            skipped += 1
            continue

        samples.append({
            'keypoints': kps,
            'adj':       ADJ,
            'label':     1,    # all correct reps in UCO
            'exercise':  f"uco_ex{sample.get('exercise', '00')}",
            'subject':   f"uco_folder{sample.get('folder', '0')}",
            'source':    'uco',
            'side':      sample.get('side', 'unknown'),
        })

    print(f"UCO loaded: {len(samples)} samples ({skipped} skipped)")
    return samples


# ─── master dataset builder ───────────────────────────────────────────────────

def build_master_dataset(
    lower_limb_only: bool = True,
    use_uco: bool = True,
) -> List[dict]:
    """
    Combine UI-PRMD + UCO into one list.
    This is the input to the PyTorch Dataset.
    """
    samples = load_uiprmd_dataset(lower_limb_only=lower_limb_only)
    if use_uco:
        samples += load_uco_dataset()

    print(f"\nMaster dataset: {len(samples)} total samples")
    print(f"  Correct:   {sum(s['label']==1 for s in samples)}")
    print(f"  Incorrect: {sum(s['label']==0 for s in samples)}")
    print(f"  Sources:   { {s['source'] for s in samples} }")
    return samples


# ─── PyTorch Dataset ─────────────────────────────────────────────────────────

class RehabDataset(Dataset):
    """
    PyTorch Dataset wrapping the master sample list.
    Returns tensors ready for ST-GCN → Transformer.
    """

    def __init__(self, samples: List[dict], augment: bool = False):
        self.samples = samples
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s   = self.samples[idx]
        kps = s['keypoints'].copy()   # (150, 6, 3)

        if self.augment:
            kps = self._augment(kps)

        # ST-GCN expects (C, T, V) = (3, 150, 6)
        kps_tensor = torch.from_numpy(kps).permute(2, 0, 1).float()   # (3, 150, 6)
        adj_tensor = torch.from_numpy(s['adj']).float()                # (6, 6)
        label      = torch.tensor(s['label'], dtype=torch.long)

        return {
            'keypoints': kps_tensor,   # (3, 150, 6)
            'adj':       adj_tensor,   # (6, 6)
            'label':     label,
            'exercise':  s['exercise'],
            'subject':   s['subject'],
            'source':    s['source'],
        }

    def _augment(self, kps: np.ndarray) -> np.ndarray:
        """
        On-the-fly augmentation.
        kps: (150, 6, 3)
        """
        # 1. temporal jitter ±20%
        if np.random.rand() < 0.5:
            factor  = np.random.uniform(0.8, 1.2)
            new_len = int(TARGET_LEN * factor)
            new_len = max(new_len, MIN_FRAMES)
            stretched = np.zeros((new_len, N_JOINTS, 3), dtype=np.float32)
            for j in range(N_JOINTS):
                for c in range(3):
                    stretched[:, j, c] = _interpolate_to_length(kps[:, j, c], new_len)
            kps = np.zeros((TARGET_LEN, N_JOINTS, 3), dtype=np.float32)
            for j in range(N_JOINTS):
                for c in range(3):
                    kps[:, j, c] = _interpolate_to_length(stretched[:, j, c], TARGET_LEN)

        # 2. mirror flip (swap L↔R)
        if np.random.rand() < 0.5:
            # swap: L_hip(0)↔R_hip(3), L_knee(1)↔R_knee(4), L_ankle(2)↔R_ankle(5)
            kps[:, [0,1,2,3,4,5], :] = kps[:, [3,4,5,0,1,2], :]
            kps[:, :, 0] *= -1.0   # flip x axis

        # 3. Gaussian noise σ=2mm (positions already normalised — use 0.02)
        if np.random.rand() < 0.5:
            kps += np.random.randn(*kps.shape).astype(np.float32) * 0.02

        # 4. small rotation around vertical axis (±10°)
        if np.random.rand() < 0.3:
            theta = np.radians(np.random.uniform(-10, 10))
            cos_t, sin_t = np.cos(theta), np.sin(theta)
            x = kps[:, :, 0].copy()
            z = kps[:, :, 2].copy()
            kps[:, :, 0] = cos_t * x - sin_t * z
            kps[:, :, 2] = sin_t * x + cos_t * z

        return kps


# ─── LOSO split ───────────────────────────────────────────────────────────────

def loso_split(
    samples: List[dict],
    held_out_subject: str
) -> Tuple[List[dict], List[dict]]:
    """
    Leave-One-Subject-Out split.
    Use 'uiprmd_s01' .. 'uiprmd_s10' as held_out_subject.
    UCO samples always go to train (no subject overlap with UI-PRMD).
    """
    train = [s for s in samples if s['subject'] != held_out_subject]
    test  = [s for s in samples if s['subject'] == held_out_subject]
    print(f"LOSO held-out: {held_out_subject} → train={len(train)}, test={len(test)}")
    return train, test


def get_all_uiprmd_subjects(samples: List[dict]) -> List[str]:
    return sorted({s['subject'] for s in samples if s['source'] == 'uiprmd'})


# ─── scalar feature extractor (XGBoost input) ────────────────────────────────

def extract_scalar_features(kps: np.ndarray) -> np.ndarray:
    """
    Compute 10 scalar features from (TARGET_LEN, 6, 3) keypoints.
    These go directly to XGBoost, bypassing ST-GCN.

    Features:
      0  L knee ROM (degrees, computed from positions)
      1  R knee ROM
      2  L knee peak angle
      3  R knee peak angle
      4  L/R ROM symmetry index (%)
      5  max angular velocity (°/s)
      6  mean jerk proxy (position jerk norm)
      7  trunk lean proxy (hip-spine deviation at peak)
      8  L/R temporal lag (frames)
      9  smoothness proxy (1 / position jerk std)
    """
    def _angle_from_positions(a, b, c):
        """Angle at joint b given positions a, b, c. Returns degrees array."""
        ba = a - b
        bc = c - b
        cos_ang = np.sum(ba * bc, axis=1) / (
            np.linalg.norm(ba, axis=1) * np.linalg.norm(bc, axis=1) + 1e-6
        )
        return np.degrees(np.arccos(np.clip(cos_ang, -1, 1)))

    # joint indices: L_hip=0, L_knee=1, L_ankle=2, R_hip=3, R_knee=4, R_ankle=5
    l_knee_ang = _angle_from_positions(kps[:, 0, :], kps[:, 1, :], kps[:, 2, :])
    r_knee_ang = _angle_from_positions(kps[:, 3, :], kps[:, 4, :], kps[:, 5, :])

    l_rom = float(np.max(l_knee_ang) - np.min(l_knee_ang))
    r_rom = float(np.max(r_knee_ang) - np.min(r_knee_ang))
    l_peak = float(np.min(l_knee_ang))
    r_peak = float(np.min(r_knee_ang))
    sym   = float(abs(l_rom - r_rom) / ((l_rom + r_rom) / 2.0 + 1e-6) * 100.0)

    # angular velocity proxy (mean frame-to-frame angle change)
    l_vel    = np.abs(np.diff(l_knee_ang)) * 30.0   # FPS=30
    max_vel  = float(np.max(l_vel))

    # jerk proxy (norm of 2nd derivative of hip position)
    hip_mid  = (kps[:, 0, :] + kps[:, 3, :]) / 2.0
    jerk_vec = np.diff(np.diff(hip_mid, axis=0), axis=0)
    jerk_norm= np.linalg.norm(jerk_vec, axis=1)
    mean_jerk= float(np.mean(jerk_norm))

    # trunk lean proxy: hip midpoint x-deviation at peak knee bend
    peak_idx    = int(np.argmin(l_knee_ang))
    trunk_lean  = float(abs(hip_mid[peak_idx, 0] - hip_mid[0, 0]))

    # temporal lag between L and R knee angle curves
    corr  = np.correlate(l_knee_ang - l_knee_ang.mean(),
                         r_knee_ang - r_knee_ang.mean(), mode='full')
    lag   = float(abs(np.argmax(corr) - (len(l_knee_ang) - 1)))

    smoothness = float(1.0 / (np.std(jerk_norm) + 1e-6))

    return np.array([
        l_rom / 180.0,
        r_rom / 180.0,
        l_peak / 180.0,
        r_peak / 180.0,
        sym / 100.0,
        max_vel / 200.0,
        mean_jerk / 10.0,
        trunk_lean,
        lag / TARGET_LEN,
        np.clip(smoothness / 100.0, 0, 1),
    ], dtype=np.float32)


# ─── ST-GCN block ─────────────────────────────────────────────────────────────

class STGCNBlock(nn.Module):
    """
    One ST-GCN block: spatial GCN + temporal conv.
    Input : (B, C_in, T, V)
    Output: (B, C_out, T, V)
    """

    def __init__(self, c_in: int, c_out: int, kernel_size: int = 9, stride: int = 1):
        super().__init__()
        pad = (kernel_size - 1) // 2

        self.gcn = nn.Linear(c_in, c_out)   # spatial (applied per node)
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(c_out),
            nn.ReLU(),
            nn.Conv2d(c_out, c_out, kernel_size=(kernel_size, 1),
                      stride=(stride, 1), padding=(pad, 0)),
            nn.BatchNorm2d(c_out),
            nn.Dropout(0.1),
        )
        self.residual = (
            nn.Sequential(
                nn.Conv2d(c_in, c_out, kernel_size=1, stride=(stride, 1)),
                nn.BatchNorm2d(c_out),
            ) if c_in != c_out or stride != 1 else nn.Identity()
        )
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
        # x: (B, C, T, V),  A: (V, V) or (B, V, V)
        B, C, T, V = x.shape

        # ensure A is always (V, V) — take first element if batched
        if A.dim() == 3:
            A = A[0]   # (V, V)

        # spatial GCN: aggregate neighbour features via A
        # reshape to (B*T, V, C), multiply by A, apply linear
        x_s = x.permute(0, 2, 3, 1).contiguous().view(B * T, V, C)   # (B*T, V, C)
        x_s = torch.bmm(A.unsqueeze(0).expand(B * T, -1, -1), x_s)   # (B*T, V, C)
        x_s = self.gcn(x_s)                                            # (B*T, V, C_out)
        x_s = x_s.view(B, T, V, -1).permute(0, 3, 1, 2)               # (B, C_out, T, V)

        # temporal conv
        out = self.tcn(x_s) + self.residual(x)
        return self.relu(out)


class STGCN(nn.Module):
    """
    3-block ST-GCN spatial encoder.
    Input : (B, 3, T, 6)    — (batch, xyz, frames, joints)
    Output: (B, T, 128)     — frame-level features for Transformer
    """

    def __init__(self):
        super().__init__()
        self.blocks = nn.ModuleList([
            STGCNBlock(3,   32,  kernel_size=9),
            STGCNBlock(32,  64,  kernel_size=9),
            STGCNBlock(64, 128,  kernel_size=9),
        ])
        self.bn_input = nn.BatchNorm1d(3 * 6)   # normalise input channels×joints

    def forward(self, x: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
        # x: (B, 3, T, 6),  A: (6, 6)
        B, C, T, V = x.shape

        # input BN
        x_flat = x.permute(0, 2, 1, 3).contiguous().view(B * T, C * V)
        x_flat = self.bn_input(x_flat)
        x = x_flat.view(B, T, C, V).permute(0, 2, 1, 3)   # back to (B, C, T, V)

        for block in self.blocks:
            x = block(x, A)   # (B, 128, T, 6)

        # pool over joints → frame-level features
        x = x.mean(dim=3)          # (B, 128, T)
        x = x.permute(0, 2, 1)    # (B, T, 128)
        return x


# ─── Two-Layer Clinical Transformer ──────────────────────────────────────────

class ClinicalTransformer(nn.Module):
    """
    Two-layer transformer with explicit clinical interpretation per layer.

    Layer 1 — Temporal correlation:
      Self-attention across T=150 frames.
      Learns which frames are related (e.g. frame 45 ↔ frame 120 = both peaks).
      Input: frame-level ST-GCN features (B, T, 128).

    Layer 2 — Clinical interpretation:
      Self-attention over the temporally-aware representations.
      Learns what the temporal correlations mean clinically
      (e.g. 'peak valgus coincides with peak velocity = dangerous pattern').
      CLS token aggregates the full rep into a single vector.

    Output: (B, 128) rep-level summary vector → XGBoost / quality head.
    """

    def __init__(
        self,
        d_model: int = 128,
        n_heads: int = 4,
        d_ff: int = 256,
        dropout: float = 0.1,
    ):
        super().__init__()

        # positional encoding
        self.pos_enc = self._build_pos_enc(TARGET_LEN + 1, d_model)  # +1 for CLS

        # CLS token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

        # Layer 1: temporal correlation
        self.layer1 = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True,
            norm_first=True,   # pre-norm for stability
        )
        self.norm1 = nn.LayerNorm(d_model)

        # Layer 2: clinical interpretation
        self.layer2 = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True,
            norm_first=True,
        )
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def _build_pos_enc(max_len: int, d_model: int) -> nn.Parameter:
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() *
                        (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        return nn.Parameter(pe.unsqueeze(0), requires_grad=False)  # (1, max_len, d_model)

    def forward(
        self,
        x: torch.Tensor,
        return_attention: bool = False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        x: (B, T, 128) ST-GCN frame features
        Returns: (B, 128) CLS vector, optionally layer-2 attention weights
        """
        B, T, D = x.shape

        # prepend CLS token
        cls  = self.cls_token.expand(B, -1, -1)        # (B, 1, 128)
        x    = torch.cat([cls, x], dim=1)               # (B, T+1, 128)
        x    = x + self.pos_enc[:, :T + 1, :]
        x    = self.dropout(x)

        # Layer 1: temporal correlation
        x    = self.layer1(x)
        x    = self.norm1(x)

        # Layer 2: clinical interpretation
        # hook attention weights for therapist dashboard heatmap
        attn_weights = None
        if return_attention:
            # manually run attention to extract weights
            x2, attn_weights = self._layer2_with_attn(x)
            x = self.norm2(x2)
        else:
            x = self.layer2(x)
            x = self.norm2(x)

        # CLS token = rep summary
        cls_out = x[:, 0, :]   # (B, 128)
        return cls_out, attn_weights

    def _layer2_with_attn(
        self, x: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Run layer2 and return (output, attention_weights)."""
        # access layer2 self-attention directly
        sa  = self.layer2.self_attn
        x_n = self.layer2.norm1(x) if self.layer2.norm_first else x
        attn_out, attn_w = sa(x_n, x_n, x_n, need_weights=True, average_attn_weights=True)
        # continue through the rest of the encoder layer manually
        x2  = x + self.layer2.dropout1(attn_out)
        x_n2 = self.layer2.norm2(x2) if self.layer2.norm_first else x2
        ff_out = self.layer2.linear2(
            self.layer2.dropout(self.layer2.activation(self.layer2.linear1(x_n2)))
        )
        x2 = x2 + self.layer2.dropout2(ff_out)
        return x2, attn_w   # attn_w: (B, T+1, T+1)


# ─── full model ───────────────────────────────────────────────────────────────

class RehabNet(nn.Module):
    """
    Full pipeline: ST-GCN → Two-Layer Clinical Transformer → Classification head.
    For XGBoost head: call encode() to get (B, 128) embeddings.
    """

    def __init__(self, n_classes: int = 2):
        super().__init__()
        self.stgcn       = STGCN()
        self.transformer = ClinicalTransformer()
        self.classifier  = nn.Sequential(
            nn.Linear(128 + 10, 64),   # 128 CLS + 10 scalar features
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, n_classes),
        )

    def encode(
        self,
        keypoints: torch.Tensor,
        adj: torch.Tensor,
        return_attention: bool = False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        Returns (B, 128) rep embedding + optional attention weights.
        adj can be (6,6) or (B,6,6) — reduced to (6,6) inside STGCNBlock.
        """
        if adj.dim() == 3:
            adj = adj[0]   # (6, 6)
        frame_feats = self.stgcn(keypoints, adj)
        cls_vec, attn = self.transformer(frame_feats, return_attention)
        return cls_vec, attn

    def forward(
        self,
        keypoints: torch.Tensor,
        adj: torch.Tensor,
        scalars: torch.Tensor,
    ) -> torch.Tensor:
        """
        keypoints : (B, 3, T, 6)
        adj       : (B, 6, 6) or (6, 6)
        scalars   : (B, 10)
        Returns   : (B, n_classes) logits
        """
        cls_vec, _ = self.encode(keypoints, adj)
        combined   = torch.cat([cls_vec, scalars], dim=1)
        return self.classifier(combined)


# ─── DataLoader builder ───────────────────────────────────────────────────────

def build_dataloaders(
    train_samples: List[dict],
    test_samples:  List[dict],
    batch_size: int = 32,
) -> Tuple[DataLoader, DataLoader]:
    """Build train/test DataLoaders with scalar features attached."""

    # attach scalar features to each sample
    for s in train_samples + test_samples:
        s['scalars'] = extract_scalar_features(s['keypoints'])

    train_ds = RehabDataset(train_samples, augment=True)
    test_ds  = RehabDataset(test_samples,  augment=False)

    # custom collate to handle scalar features
    def collate(batch):
        return {
            'keypoints': torch.stack([b['keypoints'] for b in batch]),
            'adj':       torch.stack([b['adj']       for b in batch]),
            'label':     torch.stack([b['label']     for b in batch]),
            'scalars':   torch.tensor(
                             np.stack([b['scalars'] for b in batch]),
                             dtype=torch.float32
                         ) if 'scalars' in batch[0] else None,
            'exercise':  [b['exercise'] for b in batch],
            'subject':   [b['subject']  for b in batch],
        }

    train_dl = DataLoader(train_ds, batch_size=batch_size,
                          shuffle=True,  collate_fn=collate, num_workers=2)
    test_dl  = DataLoader(test_ds,  batch_size=batch_size,
                          shuffle=False, collate_fn=collate, num_workers=2)
    return train_dl, test_dl


# ─── training loop ────────────────────────────────────────────────────────────

def train_one_epoch(
    model: RehabNet,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> float:
    model.train()
    total_loss = 0.0
    adj_shared = torch.from_numpy(ADJ).to(device)

    for batch in loader:
        kps     = batch['keypoints'].to(device)   # (B, 3, 150, 6)
        labels  = batch['label'].to(device)
        scalars = batch['scalars'].to(device) if batch['scalars'] is not None \
                  else torch.zeros(kps.shape[0], 10, device=device)

        logits = model(kps, adj_shared, scalars)
        loss   = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(
    model: RehabNet,
    loader: DataLoader,
    device: torch.device,
) -> dict:
    model.eval()
    adj_shared = torch.from_numpy(ADJ).to(device)
    all_preds, all_labels = [], []

    for batch in loader:
        kps     = batch['keypoints'].to(device)
        labels  = batch['label'].to(device)
        scalars = batch['scalars'].to(device) if batch['scalars'] is not None \
                  else torch.zeros(kps.shape[0], 10, device=device)

        logits = model(kps, adj_shared, scalars)
        preds  = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    from sklearn.metrics import accuracy_score, f1_score, classification_report
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro')
    print(classification_report(all_labels, all_preds,
                                 target_names=['incorrect', 'correct']))
    return {'accuracy': acc, 'f1_macro': f1}


# ─── LOSO training runner ─────────────────────────────────────────────────────

def run_loso_training(
    n_epochs: int = 30,
    batch_size: int = 32,
    lr: float = 1e-3,
    device_str: str = 'cuda' if torch.cuda.is_available() else 'cpu',
):
    """
    Full LOSO cross-validation run.
    Trains one model per UI-PRMD subject, reports mean ± std accuracy.
    """
    device  = torch.device(device_str)
    samples = build_master_dataset(lower_limb_only=True, use_uco=True)
    subjects = get_all_uiprmd_subjects(samples)

    results = []
    for subj in subjects:
        print(f"\n{'='*50}")
        print(f"LOSO fold: held-out = {subj}")
        train_s, test_s = loso_split(samples, subj)
        train_dl, test_dl = build_dataloaders(train_s, test_s, batch_size)

        model     = RehabNet(n_classes=2).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

        for epoch in range(n_epochs):
            loss = train_one_epoch(model, train_dl, optimizer, device)
            scheduler.step()
            if (epoch + 1) % 10 == 0:
                print(f"  Epoch {epoch+1}/{n_epochs}  loss={loss:.4f}")

        metrics = evaluate(model, test_dl, device)
        results.append(metrics)
        print(f"  → acc={metrics['accuracy']:.3f}  f1={metrics['f1_macro']:.3f}")

    accs = [r['accuracy'] for r in results]
    print(f"\nLOSO summary: acc = {np.mean(accs):.3f} ± {np.std(accs):.3f}")
    return results


# ─── smoke test ───────────────────────────────────────────────────────────────

if __name__ == '__main__':
    print("── Architecture smoke test (no data needed) ────────────────")
    device = torch.device('cpu')
    B = 4

    kps     = torch.randn(B, 3, TARGET_LEN, N_JOINTS)
    adj     = torch.from_numpy(ADJ).unsqueeze(0).expand(B, -1, -1)
    scalars = torch.randn(B, 10)

    model  = RehabNet(n_classes=2)
    logits = model(kps, adj, scalars)
    print(f"  Input  kps:     {kps.shape}")
    print(f"  Input  adj:     {adj.shape}")
    print(f"  Input  scalars: {scalars.shape}")
    print(f"  Output logits:  {logits.shape}")

    # test attention extraction
    cls_vec, attn = model.encode(kps, adj, return_attention=True)
    print(f"  CLS embedding:  {cls_vec.shape}")
    print(f"  Attn weights:   {attn.shape if attn is not None else None}")

    # adjacency matrix
    print(f"\n── Adjacency matrix (normalised) ───────────────────────────")
    print(np.round(ADJ, 3))

    print("\n── Smoke test passed ───────────────────────────────────────")
    print("\nTo load data and run LOSO, call:")
    print("  results = run_loso_training(n_epochs=30, batch_size=32)")

── Architecture smoke test (no data needed) ────────────────
  Input  kps:     torch.Size([4, 3, 150, 6])
  Input  adj:     torch.Size([4, 6, 6])
  Input  scalars: torch.Size([4, 10])
  Output logits:  torch.Size([4, 2])
  CLS embedding:  torch.Size([4, 128])
  Attn weights:   torch.Size([4, 151, 151])

── Adjacency matrix (normalised) ───────────────────────────
[[0.333 0.289 0.    0.333 0.    0.   ]
 [0.289 0.25  0.289 0.    0.25  0.   ]
 [0.    0.289 0.333 0.    0.    0.333]
 [0.333 0.    0.    0.333 0.289 0.   ]
 [0.    0.25  0.    0.289 0.25  0.289]
 [0.    0.    0.333 0.    0.289 0.333]]

── Smoke test passed ───────────────────────────────────────

To load data and run LOSO, call:
  results = run_loso_training(n_epochs=30, batch_size=32)


In [2]:
# In Colab — paste this as a new cell
results = run_loso_training(
    n_epochs=10,      # short first run to verify loss is decreasing
    batch_size=32,
    lr=1e-3,
)

UI-PRMD loaded: 1180 samples
UCO loaded: 212 samples (4 skipped)

Master dataset: 1392 total samples
  Correct:   802
  Incorrect: 590
  Sources:   {'uiprmd', 'uco'}

LOSO fold: held-out = uiprmd_s01
LOSO held-out: uiprmd_s01 → train=1272, test=120
  Epoch 10/10  loss=0.4254
              precision    recall  f1-score   support

   incorrect       0.76      0.63      0.69        60
     correct       0.69      0.80      0.74        60

    accuracy                           0.72       120
   macro avg       0.72      0.72      0.71       120
weighted avg       0.72      0.72      0.71       120

  → acc=0.717  f1=0.715

LOSO fold: held-out = uiprmd_s02
LOSO held-out: uiprmd_s02 → train=1272, test=120
  Epoch 10/10  loss=0.4438
              precision    recall  f1-score   support

   incorrect       0.88      0.35      0.50        60
     correct       0.59      0.95      0.73        60

    accuracy                           0.65       120
   macro avg       0.73      0.65      0.62  

In [3]:
results = run_loso_training(
    n_epochs=30,
    batch_size=32,
    lr=1e-3,
)

UI-PRMD loaded: 1180 samples
UCO loaded: 212 samples (4 skipped)

Master dataset: 1392 total samples
  Correct:   802
  Incorrect: 590
  Sources:   {'uco', 'uiprmd'}

LOSO fold: held-out = uiprmd_s01
LOSO held-out: uiprmd_s01 → train=1272, test=120
  Epoch 10/30  loss=0.4718


KeyboardInterrupt: 